# Thermal face respiration research: a step-by-step walkthrough
**Presentation edition · prepared 17 October 2026**

**Research question:** Can changes in thermal intensity around the nose estimate breathing rate for a person unseen during training?

This notebook tells the development story in logical sequence: initial signal processing → data corrections → neural pilots → fair comparison → limitations and next work. The repository does not provide a complete dated experiment log, so stage numbers indicate development order, **not invented dates or elapsed training times**.

**Evidence labels used throughout**
- **Executed now:** outputs produced by running the visible cell while building this notebook.
- **Historical/documented:** values transcribed from `README.md`, `docs/research_strategy.md`, or legacy reports; not freshly reproduced measurements.
- **Synthetic demonstration:** invented signals used only to explain and verify mechanics, never participant results.
- **Conditional reproduction:** real-data work runs only when inputs exist and the explicit rerun switch is enabled.

The delivered notebook contains executed outputs and can be presented without rerunning it. A companion HTML edition is suitable for browser presentation. This local edition embeds participant thermal frames and supplied processing maps; review dataset sharing terms before distributing it.

**Updated 18 October 2026:** Section 14 integrates the complete CSRT experiment, followed by revised limitations and next steps. Original and tracking-run outputs are preserved from their actual executions; this integration does not retrain the models. Cell numbers indicate presentation order. The earlier sections describe the fixed-crop baseline.





**Follow-up on 31 October 2026:** Section 16 evaluates 12-second supported segments after the original 20-second MRF comparison yielded no eligible windows. The follow-up is exploratory and does not replace the original result.


**Project aim:** build a thermal-video breathing-rate detector with breathing-region segmentation. Kwon et al. guides the segmentation branch, and JoulesEye guides tracking. Section 15 records the corrected Kwon-style implementation and original 20-second experiment; Section 16 is an exploratory short-segment follow-up.


## 1. What this project does
Breathing produces temporal changes around the nose in thermal imagery. The current pipeline summarizes nine nasal patches, preserves timestamps, builds windows, and predicts breaths per minute (BPM).

`Thermal frames + CSV timestamps → fixed nose crop → 9 intensity traces → 10 Hz resampling + missing-data channel → 20 s windows → CNN / CNN + GRU → breathing rate`

The broader ambition includes recognizing identity, but the implemented pilot is respiration regression. It is not a validated face recognition or clinical monitoring system. The available images are 8-bit grayscale intensities, not calibrated degrees Celsius.

### Research foundations
| Paper | Role in this project |
|---|---|
| [Coşar et al., 2018, Thermal Camera Based Physiological Monitoring with an Assistive Robot](https://doi.org/10.1109/EMBC.2018.8513201) | Dataset source according to project notes; facial thermal signals and Fourier analysis |
| [Kwon et al., 2023, Breathing-Associated Facial Region Segmentation for Thermal Camera-Based Indirect Breathing Monitoring](https://doi.org/10.1109/JTEHM.2023.3295775) | BAFR segmentation inspiration; the PDF included in the repository |
| [Adhikary et al., 2023, JoulesEye](https://doi.org/10.1145/3631422) | Supporting ideas for tracking and handling motion/occlusion |

These are different studies and datasets. Kwon's BIOPAC reference is not the reference used by this project's dataset. JoulesEye's energy-expenditure error is not respiration-rate error. The current CNN–GRU is this repository's exploratory implementation, not an exact reproduction of these papers.

## 2. Environment, provenance, and input availability
Run from the repository root or its `docs` directory. Standard processing needs NumPy and Pillow; neural experiments need PyTorch. Notebook display uses IPython and Matplotlib. Install the project's `requirements.txt` in your chosen environment; install notebook/display packages separately if needed.

This edition uses the restored data in `Manuel_clean/`. `RERUN_EXPERIMENTS=True` executes extraction, MLP, and unified evaluation. Set it to `False` before using Run All during a presentation to avoid retraining; saved outputs remain viewable without execution. New reports use notebook-specific filenames. A GPU is optional; `auto` selects CUDA if available.

In [ ]:
from pathlib import Path
import sys, json, hashlib, platform, importlib.util
import numpy as np
from IPython.display import display, Markdown, Image

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'thermal_face' / 'data.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA_ROOT = ROOT / 'Manuel_clean'
RERUN_EXPERIMENTS = True
DEVICE, EPOCHS, SEED = 'auto', 40, 7
TRACE_DIR = ROOT / 'artifacts' / 'traces'
print('Python:', platform.python_version())
print('NumPy:', np.__version__)
print('Retraining enabled:', RERUN_EXPERIMENTS)
from thermal_face.data import SUBJECTS
availability = []
for person in SUBJECTS:
    frames = len(list((DATA_ROOT / person).glob('frame_*.png')))
    csvs = len(list(DATA_ROOT.glob(f'*{person}_respiration.csv')))
    availability.append((person, frames, csvs))
    print(f'{person:8} | frames present: {frames:5} | respiration CSV files: {csvs}')
RAW_AVAILABLE = all(n > 0 and c == 1 for _, n, c in availability)
print('Complete original dataset available:', RAW_AVAILABLE)
print('Saved trace directory available:', TRACE_DIR.is_dir())
print('PyTorch installed:', importlib.util.find_spec('torch') is not None)

In [ ]:
# Source fingerprints identify the implementation inspected by this presentation.
for relative in ['README.md', 'docs/research_strategy.md',
                 'thermal_face/features.py', 'thermal_face/windows.py',
                 'thermal_face/models.py', 'thermal_face/train_rate.py',
                 'thermal_face/evaluate.py']:
    digest = hashlib.sha256((ROOT / relative).read_bytes()).hexdigest()
    print(relative, digest)

## 3. First approach: the legacy image-processing pipeline
The archived scripts preserve an earlier approach inspired by breathing-associated facial regions:

| Order | Operation | Source |
|---|---|---|
| 1 | Load previously extracted 8-bit PNG frames | `legacy/step2_step3.py` |
| 2–3 | Detect frozen runs; derive a perinasal crop from a warm face region | `legacy/step2_step3.py`, `legacy/step3_roi.py` |
| 4 | KLT optical-flow stabilization | `legacy/step4_5_6.py` |
| 5–6 | Rolling pixel standard deviation; normalize to a breathing association map | `legacy/step4_5_6.py` |
| 7 | MRF/ICM segmentation of the averaged map | `legacy/step7_8_9_10.py` |
| 8–10 | Masked waveform, bandpass filtering, peak counting, summaries | `legacy/step7_8_9_10.py` |
| Evaluation | Compare peak/FFT rates with reference events | `legacy/gt_comparison.py` |

BAF in Kwon equation 1 is sample standard deviation. The legacy square root is therefore appropriate; the substantive departures are reversed stabilization, whole-recording static masking, and inconsistent timing/signal extraction. Historical scripts contain machine-specific paths and are not executed here.

In [ ]:
# Executed now: read the archived output. Its claims are corrected immediately below.
legacy_text = (ROOT / 'legacy' / 'gt_comparison_report.txt').read_text(encoding='utf-8')
print('HISTORICAL OUTPUT — known labeling/timing problems; not a valid benchmark')
print(legacy_text)

## 4. Why the first result was not a trustworthy benchmark
The historical peak-count MAE was **15.40 BPM**, but it cannot be compared directly with the corrected neural evaluation:

- Actual CSV spacing is about 27 Hz; old processing assumed 25 FPS and removed/renumbered frozen frames, compressing time.
- A broad face crop could include chest pixels or poorly localize the nose.
- Counting both markers 1 and 2 approximately doubled Joao's reference rate. The current code counts marker 1 only; the semantics still need independent confirmation.
- The legacy report calls the reference BIOPAC/chest belt. Project research notes identify manually annotated inhalation times instead.
- A 256-sample spectral segment at 25 Hz has approximately 5.86 BPM bin spacing, limiting precision.
- `anestis` has unequal frame and reference-row counts.

**Presenter note:** We corrected the measurement protocol before interpreting model accuracy. Do not claim an improvement “from 15.40 to 2.986” because these are different, non-comparable evaluations.

In [ ]:
from datetime import datetime, timedelta
from thermal_face.data import breath_events, recording_rate
# Synthetic marker example: three inhalation events and three other-phase events.
times = [datetime(2020, 1, 1) + timedelta(seconds=s) for s in range(10)]
markers = [0, 1, 2, 0, 1, 2, 0, 1, 2, 0]
print('Synthetic example; not participant data')
print('Correct marker-1 event count:', len(breath_events(times, markers)))
print('Full-duration marker-1 rate:', recording_rate(times, markers), 'BPM')
print('Incorrect all-nonzero rate:', 60 * sum(m != 0 for m in markers) / 9, 'BPM')
print('Legacy FFT bin spacing:', 60 * 25 / 256, 'BPM')

## 5. Audit recordings before building training examples
The table below is a **historical transcription** of the research notes. The following code independently audits the restored files for comparison. Matching row counts are necessary evidence, not proof of timestamp synchronization.

| Person | Frames | CSV rows | Marker-1 events | Full-CSV rate (BPM) | Current use |
|---|---:|---:|---:|---:|---|
| Joao | 3036 | 3036 | 37 | 18.22 | provisional alignment |
| anestis | 2863 | 3526 | 44 | 21.72 | excluded until aligned |
| Claudio | 2903 | 2903 | 38 | 18.62 | provisional alignment |
| Manuel | 2700 | 2700 | 38 | 18.68 | provisional alignment |
| Jaime | 2652 | 2652 | 43 | 21.24 | provisional alignment |

Full-recording rate uses `60 × event count / recording duration`. Later window labels use `60 / median inter-inhalation interval`; these are related but different statistics.

In [ ]:
from thermal_face.data import audit
if RAW_AVAILABLE:
    audit_rows = audit(DATA_ROOT)
    print(json.dumps(audit_rows, indent=2))
else:
    print('NOT RECOMPUTED: original frames and respiration CSVs are absent/incomplete.')
    print('Reproduction command: python -m thermal_face.data --out artifacts/data_audit.json')

## 6. Corrected feature extraction: retain time, isolate the nose
`config/nose_boxes.json` contains provisional, person-specific manually inspected crops. Divide each crop into a **3 × 3 grid** and average intensity within each patch, producing nine traces per recording. Flag identical-frame runs of at least five frames as missing, while retaining original CSV elapsed time. Skip mismatched recordings.

These fixed crops are not an automatic detector or tracker. Head motion can move the nose outside the box. The current neural path uses these traces; it does not consume legacy MRF masks or full-frame spatial CNN features.

In [ ]:
boxes = json.loads((ROOT / 'config' / 'nose_boxes.json').read_text())
print('Configured crops [x, y, width, height]:')
print(json.dumps(boxes, indent=2))
from thermal_face.features import extract
if RERUN_EXPERIMENTS and RAW_AVAILABLE:
    extract(DATA_ROOT, TRACE_DIR, ROOT / 'config' / 'nose_boxes.json')
else:
    print('Extraction not run: restore inputs and enable RERUN_EXPERIMENTS.')
    print('Reproduction command: python -m thermal_face.features --out artifacts/traces')

## 7. Resample, normalize, and build supervised windows
- Resample to **10 Hz**, using valid timestamps as interpolation anchors.
- Mark interpolation gaps over **0.25 s** with a tenth, missing-data channel; reject windows with more than 20% missing samples.
- Normalize each nasal channel by its window median and standard deviation (with a scale floor of 1).
- Build **20-second windows every 5 seconds**, each shaped `(10, 200)`.
- Label each window with `60 / median(diff(marker-1 event times))`. Require at least four events, intervals from 0.8–10 s, and rates from 6–45 BPM.
- Exclude `anestis` until alignment is resolved. Treat the first 60 s as still, later windows as moving, and boundary-crossing windows as transition; this is a protocol assumption.

The following cell runs the **actual preprocessing functions on a synthetic sinusoid** to illustrate their behavior. It is not evidence of real-data accuracy.

In [ ]:
from thermal_face.windows import resample_signal, reference_rate, condition_for, load_window_records
from thermal_face.baselines import spectral_rate
t = np.arange(0, 21, 1 / 27)
traces = np.stack([100 + (4 + j / 4) * np.sin(2 * np.pi * 0.3 * t)
                   for j in range(9)], axis=1)
valid = np.ones(len(t), dtype=bool)
target = np.arange(0, 20, 0.1)
x_demo, missing, reason = resample_signal(t, traces, valid, target)
events = np.arange(0.5, 20, 60 / 18)
label, label_reason = reference_rate(events, 0, 20)
estimated, spectral_reason = spectral_rate(x_demo)
print('SYNTHETIC 18 BPM signal')
print('Input shape:', x_demo.shape, '| missing fraction:', missing)
print('Reference:', label, 'BPM | spectral estimate:', estimated, 'BPM')
print('Rejection reasons:', reason, label_reason, spectral_reason)
print('Conditions:', [condition_for(a, a + 20) for a in [0, 50, 60]])
assert x_demo.shape == (10, 200) and abs(estimated - 18) < 0.75

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import io
def show_plot(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format='png', dpi=140, bbox_inches='tight')
    plt.close(fig)
    display(Image(data=buffer.getvalue()))
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(target, x_demo[0], color='#187b8d', label='Normalized nasal patch (synthetic)')
for event in events:
    ax.axvline(event, color='#cc7a00', alpha=0.4)
ax.set(xlabel='Elapsed time (seconds)', ylabel='Normalized intensity',
       title='Synthetic demonstration: thermal oscillation and illustrative event times')
ax.legend(loc='upper right')
show_plot(fig)

In [ ]:
if TRACE_DIR.is_dir() and list(TRACE_DIR.glob('*.npz')):
    records = load_window_records(TRACE_DIR)
    for person, windows in records.items():
        print(person, 'candidate windows:', len(windows),
              'eligible:', sum(w.eligible for w in windows))
else:
    print('Real window generation not recomputed: saved traces are absent.')

### Restored dataset: original frames and supplied BAP/BAFR outputs
The following montage reads the supplied files; it does not recompute the legacy segmentation. BAF denotes the rolling variability feature, BAP its normalized association map, and BAFR the segmented region. A high association value can also reflect motion and is not a calibrated probability of breathing.

The original frame and legacy maps use different coordinates and image sizes. They are displayed separately, not overlaid as though aligned. The red box on the original frame is the current fixed nose crop. Supplied masks and overlays remain historical processing outputs; they are not ground-truth labels. A reported zero tracking-fallback count is not proof of accurate tracking.

In [ ]:
from PIL import Image as PILImage
from matplotlib.patches import Rectangle
inventory = '| Person | Original | Clean | Stable | BAF | BAP |\n|---|---:|---:|---:|---:|---:|\n'
for person in SUBJECTS:
    counts = [len(list((DATA_ROOT / (person + suffix)).glob('*')))
              for suffix in ['', '_clean', '_stable', '_baf', '_bap']]
    inventory += '| ' + person + ' | ' + ' | '.join(map(str, counts)) + ' |\n'
display(Markdown(inventory))
print('Original frames actually present:', sum(n for _, n, _ in availability))
print('The supplied README_pipeline.md headline says 15,154; the actual folder counts total 14,154.')
fig, axes = plt.subplots(5, 4, figsize=(13, 13))
for row, person in enumerate(SUBJECTS):
    paths = [next(iter(sorted((DATA_ROOT / person).glob('frame_*.png')))),
             DATA_ROOT / f'{person}_bap_mean.png',
             DATA_ROOT / f'{person}_mask.png',
             DATA_ROOT / f'{person}_mask_overlay.png']
    for col, (path, title) in enumerate(zip(paths, ['Original + current crop', 'Supplied mean BAP', 'Supplied BAFR mask', 'Supplied mask overlay'])):
        with PILImage.open(path) as picture:
            axes[row, col].imshow(np.array(picture), cmap='gray')
        axes[row, col].set_title(f'{person}: {title}', fontsize=9)
        axes[row, col].axis('off')
    bx, by, bw, bh = boxes[person]
    axes[row, 0].add_patch(Rectangle((bx, by), bw, bh, fill=False, edgecolor='red', linewidth=1.4))
fig.suptitle('Restored local data — supplied segmentation outputs, not recomputed masks')
fig.tight_layout()
show_plot(fig)

### Actual extracted nasal traces and annotations
These plots use freshly extracted traces from original frames and CSV timestamps. Orange vertical lines mark marker-1 events. Shaded points indicate frozen-run samples, which are retained on the original time axis but excluded from interpolation anchors. Intensity is not calibrated temperature.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
for ax, person in zip(axes, ['Claudio', 'Jaime', 'Joao', 'Manuel']):
    with np.load(TRACE_DIR / f'{person}.npz') as trace_file:
        seconds, signal, valid, marker = [trace_file[k] for k in ['seconds', 'trace', 'valid', 'marker']]
    ax.plot(seconds, signal.mean(axis=1), linewidth=0.8, color='#187b8d')
    ax.scatter(seconds[~valid], signal.mean(axis=1)[~valid], s=3, color='red', label='Frozen-run samples')
    for event in seconds[marker == 1]:
        ax.axvline(event, color='#cc7a00', alpha=0.25)
    ax.set(title=person, ylabel='Mean intensity')
axes[-1].set_xlabel('Elapsed seconds from CSV timestamps')
axes[0].legend()
fig.suptitle('Actual data: nasal intensity and annotated inhalation events')
fig.tight_layout()
show_plot(fig)

## 8. First learned model: spectral MLP
The MLP takes normalized FFT magnitudes from the nine traces in the 0.1–0.75 Hz band, plus missing fraction. Training-only feature standardization prevents using the held-out person's distribution to fit the model. A 16-unit tanh hidden layer predicts rate; the NumPy implementation trains for 800 updates with seed 7.

**Historical finding:** macro MAE **4.80 BPM**, versus **2.50 BPM** for the training-median comparator. The MLP did not establish added value. Its result motivates investigating temporal features and localization; it does not prove that model size is the problem.

In [ ]:
if RERUN_EXPERIMENTS and TRACE_DIR.is_dir():
    from thermal_face.train_mlp import run as run_mlp
    mlp_report = run_mlp(TRACE_DIR, ROOT / 'artifacts' / 'notebook_mlp.json')
else:
    print('MLP training not rerun. Historical MAE: 4.80 BPM; median baseline: 2.50 BPM.')
    print('Reproduction command: python -m thermal_face.train_mlp --traces artifacts/traces --out artifacts/notebook_mlp.json')

## 9. Temporal CNN and CNN + GRU
The CNN learns local patterns along time. The GRU integrates the resulting sequence. Both receive **nine traces plus a missing-data channel**, not images.

| CNN + GRU stage | Output dimensions (excluding batch) |
|---|---|
| Input | 10 channels × 200 samples |
| Conv1D 10→24, kernel 9, ReLU, average pool /2 | 24 × 100 |
| Conv1D 24→32, kernel 9, ReLU, average pool /2 | 32 × 50 |
| Transpose, GRU with hidden size 32 | 50 steps × 32 features → final hidden state |
| Linear 32→16, ReLU, Linear 16→1 | one rate in BPM |

The comparison CNN adds a third convolution and averages across time before its regression head. Training uses **AdamW**, learning rate 0.001, weight decay 0.01, batch size 16, Smooth L1 loss (beta 3), 40 epochs, and seed 7. The complete window is required; GRU state resets between windows. The missing channel is a feature, not a recurrent-state mask.

In [ ]:
# Executed architecture inspection; random weights below are not trained results.
if importlib.util.find_spec('torch'):
    import torch
    from thermal_face.models import CNNRateNet, CNNGRURateNet
    print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
    for cls in [CNNRateNet, CNNGRURateNet]:
        model = cls().eval()
        with torch.no_grad():
            output = model(torch.from_numpy(x_demo[None]))
        print(cls.__name__, '| parameters:', sum(p.numel() for p in model.parameters()),
              '| output shape:', tuple(output.shape))
    print(CNNGRURateNet())
else:
    print('Architecture inspection skipped: PyTorch is not installed.')

## 10. Evaluation that keeps people separate
Use **leave-one-person-out** evaluation: train on three people, test on the fourth, and repeat. Every overlapping window from a person remains in that person's fold. Never randomly split overlapping windows across train/test.

The training-median baseline predicts the median training-window label for every held-out window. It is an essential check because the subjects' rates occupy a narrow range. The spectral baseline detrends, applies a Hann taper, and averages normalized nasal-patch power in 6–45 BPM. Eightfold zero padding gives denser frequency samples, not better underlying resolution (about 3 BPM for 20 s).

**MAE** is mean absolute predicted-minus-reference error. **Macro MAE** first computes each person's MAE, then averages across people so each person has equal weight. Pooled window MAE instead weights people by window count.

In [ ]:
people = ['Claudio', 'Jaime', 'Joao', 'Manuel']
for held_out in people:
    training = [p for p in people if p != held_out]
    assert held_out not in training
    print('Train:', ', '.join(training), '| test:', held_out)
print('Four independent held-out people; 77 overlapping windows are not 77 independent subjects.')

### Reproduce the full evaluation
This cell trains both neural architectures and compares them with spectral and median baselines. It writes fold membership, per-window predictions, source-trace hashes, environment information, exclusions, coverage, and condition summaries. `DEVICE='cuda'` explicitly requires CUDA; `auto` allows CPU.

The commands in the repository were documented as prior runs; the cell below is the actual runnable equivalent used by this notebook when enabled. No original terminal-session history is available.

In [ ]:
report = None
if RERUN_EXPERIMENTS and TRACE_DIR.is_dir() and importlib.util.find_spec('torch'):
    from time import perf_counter
    from thermal_face.evaluate import run as evaluate
    evaluation_started = perf_counter()
    report = evaluate(TRACE_DIR, ROOT / 'artifacts' / 'notebook_evaluation.json',
                      epochs=EPOCHS, seed=SEED, device=DEVICE, verbose=True)
    print('Fresh evaluation wall time (seconds):', round(perf_counter() - evaluation_started, 2))
    print(json.dumps(report['overall'], indent=2))
else:
    print('Full training/evaluation NOT rerun in this presentation build.')
    print('python -m thermal_face.evaluate --device auto --epochs 40 --seed 7 --out artifacts/notebook_evaluation.json')

### Fresh evaluation on the restored data
The following outputs are newly computed by the notebook, using the actual dataset. These are distinct from the historical comparison in section 11. Training time below is the wall time of this run, not an inferred historical duration.

In [ ]:
if report is not None:
    print('Environment:', json.dumps(report['environment'], indent=2))
    print('Protocol:', json.dumps(report['protocol'], indent=2))
    summary = report['overall']
    print('Coverage:', summary['eligible_windows'], '/', summary['candidate_windows'])
    print('Reference rejections:', summary['reference_rejections'])
    table = '| Method | Overall macro MAE | Still | Moving | Transition |\n|---|---:|---:|---:|---:|\n'
    for method, metrics in summary['common_comparison']['methods'].items():
        vals = [metrics['macro_mae_bpm']] + [report['by_condition'][c]['common_comparison']['methods'][method]['macro_mae_bpm'] for c in ['still', 'moving', 'transition']]
        table += '| ' + method + ' | ' + ' | '.join(f'{v:.3f}' if v is not None else 'N/A' for v in vals) + ' |\n'
    display(Markdown('**Fresh run: MAE in BPM**\n\n' + table))
    median_error = summary['common_comparison']['methods']['train_median']['macro_mae_bpm']
    gru_error = summary['common_comparison']['methods']['cnn_gru']['macro_mae_bpm']
    print('Fresh-run interpretation: CNN + GRU', 'beats' if gru_error < median_error else 'does not beat', 'the median baseline overall.')
    fig, axes = plt.subplots(4, 1, figsize=(11, 10), sharex=True)
    for ax, person in zip(axes, ['Claudio', 'Jaime', 'Joao', 'Manuel']):
        rows = [r for r in report['windows'] if r['subject'] == person and r['eligible']]
        centers = [(r['start_s'] + r['end_s']) / 2 for r in rows]
        ax.plot(centers, [r['reference_bpm'] for r in rows], 'o-', label='Reference', color='black')
        for method, label in [('cnn_gru', 'CNN + GRU'), ('train_median', 'Training median')]:
            ax.plot(centers, [r['predictions_bpm'][method] for r in rows], '.--', label=label)
        ax.axvline(60, color='gray', linestyle=':')
        ax.set(title=person, ylabel='BPM')
    axes[0].legend(ncol=3)
    axes[-1].set_xlabel('Window center (seconds); each point summarizes a 20-second window')
    fig.suptitle('Fresh held-out-person predictions on actual data')
    fig.tight_layout()
    show_plot(fig)
else:
    print('No fresh report in this execution; enable RERUN_EXPERIMENTS to reproduce it.')

## 11. Recorded results and interpretation
The following tables and plots use **historical values transcribed from `docs/research_strategy.md`**, for 40 epochs and seed 7. They are deliberately independent of the fresh run above. Original historical neural JSON artifacts were not supplied; new per-window predictions are saved in `artifacts/notebook_evaluation.json`.

The documented CPU and GPU runs agree at three decimal places. Using a GPU changes runtime, not evidence of model accuracy. No training-duration claim is made because complete logs are unavailable.

In [ ]:
historical = {
    'Training median': [2.501, 1.944, 2.711, 3.527],
    'Spectral': [10.618, 7.192, 14.046, 12.771],
    'CNN': [3.577, 2.787, 3.832, 5.452],
    'CNN + GRU': [2.986, 2.286, 3.573, 3.885],
}
table = '| Method | Overall | Still | Moving | Transition |\n|---|---:|---:|---:|---:|\n'
for method, values in historical.items():
    table += '| ' + method + ' | ' + ' | '.join(f'{v:.3f}' for v in values) + ' |\n'
display(Markdown('**Historical macro MAE, BPM (lower is better)**\n\n' + table))
fig, ax = plt.subplots(figsize=(9, 3.6))
names = list(historical)
values = [historical[n][0] for n in names]
bars = ax.barh(names, values, color=['#267764', '#9299a3', '#4482ad', '#b07935'])
ax.bar_label(bars, fmt='%.3f', padding=4)
ax.set(xlim=(0, 12), xlabel='Macro MAE (breaths/minute)',
       title='Historical documented results — not rerun measurements')
ax.invert_yaxis()
show_plot(fig)

### Per-person errors from the recorded pilot
| Held-out person | CNN + GRU | CNN | Median baseline |
|---|---:|---:|---:|
| Claudio | 2.565 | 4.276 | 1.956 |
| Jaime | 4.241 | 3.830 | 2.626 |
| Joao | 1.653 | 3.104 | 2.146 |
| Manuel | 3.484 | 3.097 | 3.277 |

All values are MAE in BPM. CNN + GRU improves aggregate error relative to this CNN run, but **does not beat the training-median baseline overall**. It beats that baseline only for Joao. These single-seed results do not establish reliable generalization or clinical suitability.

### Coverage and exclusions
The historical evaluation used **77/84 candidate windows (91.7%)**: still 36/36, moving 29/36, transition 12/12. Six windows had implausible reference-event intervals; one had fewer than four events. No candidate window failed signal screening in that run.

Coverage includes reference-label screening, so it is **not live system availability**. Missing labels especially affect moving-period results. Spectral error alone does not prove a particular failure cause; inspect crops and labels before drawing that conclusion.

## 12. Verification performed for this notebook
Run the repository tests and report their real output. Tests verify implementation behavior, not research validity. The source notes describe historical test passes; the output below records only what this notebook actually executes in its current environment. CUDA-dependent tests may skip when CUDA is unavailable.

In [ ]:
import subprocess, os
# Keep temporary test artifacts inside this writable checkout.
test_temp = ROOT / 'artifacts' / 'notebook_test_tmp'
test_temp.mkdir(parents=True, exist_ok=True)
test_env = dict(os.environ, TMP=str(test_temp), TEMP=str(test_temp))
test_run = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
                          cwd=ROOT, env=test_env, capture_output=True, text=True)
print(test_run.stdout)
print(test_run.stderr)
print('Test process exit code:', test_run.returncode)
if test_run.returncode:
    print('Some checks failed or dependencies are unavailable; inspect the output before claiming verification.')

## 13. Source-code appendix: implementation used by the walkthrough
The following source snapshots preserve the exact current Python implementation for explanation without opening other files. They are **reference listings**, not extra executed training cells. The executable cells above import these same modules from the repository. Legacy code remains in `legacy/`; it is not the corrected benchmark implementation.

### `thermal_face/data.py`

```python
"""Audit the original recordings before constructing supervised examples.

CSV rows are timestamped and correspond one-to-one to thermal frames for four
subjects. Anestis has extra CSV rows, so its frame timestamps are unknown.
"""

from __future__ import annotations

import argparse
import csv
import json
import statistics
from datetime import datetime
from pathlib import Path

SUBJECTS = ("Joao", "anestis", "Claudio", "Manuel", "Jaime")
TIME_FORMAT = "%Y%m%d-%H:%M:%S.%f"


def read_respiration(path: Path) -> tuple[list[datetime], list[int]]:
    times: list[datetime] = []
    markers: list[int] = []
    with path.open(newline="", encoding="utf-8-sig") as stream:
        for row in csv.reader(stream):
            if len(row) != 2:
                raise ValueError(f"Malformed CSV row in {path}: {row}")
            times.append(datetime.strptime(row[0], TIME_FORMAT))
            markers.append(int(row[1]))
    if len(times) < 2 or any(b <= a for a, b in zip(times, times[1:])):
        raise ValueError(f"Missing or non-increasing timestamps: {path}")
    return times, markers


def breath_events(times: list[datetime], markers: list[int]) -> list[datetime]:
    """Marker 1 is a breath event; Joao's marker 2 is its other phase."""
    return [t for t, marker in zip(times, markers) if marker == 1]


def recording_rate(times: list[datetime], markers: list[int]) -> float:
    events = breath_events(times, markers)
    duration = (times[-1] - times[0]).total_seconds()
    return 60.0 * len(events) / duration


def audit(root: Path) -> list[dict]:
    records = []
    for subject in SUBJECTS:
        frame_paths = sorted((root / subject).glob("frame_*.png"))
        csv_paths = list(root.glob(f"*{subject}_respiration.csv"))
        if len(csv_paths) != 1:
            raise FileNotFoundError(f"Expected one respiration CSV for {subject}")
        times, markers = read_respiration(csv_paths[0])
        intervals = [(b - a).total_seconds() for a, b in zip(times, times[1:])]
        marker_values = sorted(set(markers))
        records.append({
            "subject": subject,
            "frames": len(frame_paths),
            "csv_rows": len(times),
            "frame_csv_aligned_by_row": len(frame_paths) == len(times),
            "csv_duration_s": round((times[-1] - times[0]).total_seconds(), 3),
            "median_csv_interval_s": round(statistics.median(intervals), 4),
            "marker_values": marker_values,
            "breath_events_marker_1": len(breath_events(times, markers)),
            "reference_bpm_full_csv": round(recording_rate(times, markers), 2),
        })
    return records


def main() -> None:
    parser = argparse.ArgumentParser(description="Audit frame and reference-label alignment")
    parser.add_argument("--root", type=Path, default=Path(__file__).resolve().parents[1])
    parser.add_argument("--out", type=Path, help="Optional JSON report path")
    args = parser.parse_args()
    report = audit(args.root)
    rendered = json.dumps(report, indent=2)
    print(rendered)
    if args.out:
        args.out.parent.mkdir(parents=True, exist_ok=True)
        args.out.write_text(rendered + "\n", encoding="utf-8")


if __name__ == "__main__":
    main()

```

### `thermal_face/features.py`

```python
"""Extract timestamped nose-region traces from original thermal frames.

The five boxes in config/nose_boxes.json are provisional hand-inspected boxes
for this dataset, not an automatic face detector. No derived frame directories
are required. Frozen runs are flagged and excluded from interpolation anchors.
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

from .data import SUBJECTS, read_respiration


def extract(root: Path, out: Path, boxes_path: Path | None = None) -> None:
    import numpy as np
    from PIL import Image

    boxes_path = boxes_path or root / "config" / "nose_boxes.json"
    boxes = json.loads(boxes_path.read_text(encoding="utf-8"))
    out.mkdir(parents=True, exist_ok=True)
    for subject in SUBJECTS:
        frames = sorted((root / subject).glob("frame_*.png"))
        csv_path = next(root.glob(f"*{subject}_respiration.csv"))
        times, markers = read_respiration(csv_path)
        if len(frames) != len(times):
            print(f"Skipping {subject}: {len(frames)} frames vs {len(times)} CSV rows; no verified alignment")
            continue
        x, y, w, h = boxes[subject]
        traces = np.zeros((len(frames), 9), dtype=np.float32)
        repeated = np.zeros(len(frames), dtype=np.bool_)
        prev = None
        for i, path in enumerate(frames):
            with Image.open(path) as image:
                frame = np.asarray(image).copy()
            if frame.ndim == 3 and frame.shape[2] == 3:
                if not (np.array_equal(frame[:, :, 0], frame[:, :, 1]) and
                        np.array_equal(frame[:, :, 0], frame[:, :, 2])):
                    raise ValueError(f"Expected grayscale channels in {path}")
                frame = frame[:, :, 0]
            if frame is None or frame.ndim != 2 or frame.shape[0] < y + h or frame.shape[1] < x + w:
                raise ValueError(f"Invalid frame or nose box: {path}")
            if prev is not None:
                repeated[i] = np.array_equal(frame, prev)
            prev = frame
            crop = frame[y:y+h, x:x+w].astype(np.float32)
            for row in range(3):
                for col in range(3):
                    part = crop[row*h//3:(row+1)*h//3, col*w//3:(col+1)*w//3]
                    traces[i, row*3+col] = float(part.mean())
        # Treat a run of five or more identical frames as missing thermal data.
        valid = np.ones(len(frames), dtype=np.bool_)
        start = None
        for i in range(1, len(frames) + 1):
            same = i < len(frames) and repeated[i]
            if same and start is None:
                start = i - 1
            elif not same and start is not None:
                if i - start >= 5:
                    valid[start:i] = False
                start = None
        seconds = np.array([(t - times[0]).total_seconds() for t in times], dtype=np.float64)
        np.savez_compressed(
            out / f"{subject}.npz", seconds=seconds, trace=traces,
            valid=valid, marker=np.asarray(markers, dtype=np.int8),
        )
        print(f"{subject}: {len(frames)} frames, {np.count_nonzero(~valid)} frozen-run frames")


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--root", type=Path, default=Path(__file__).resolve().parents[1])
    parser.add_argument("--out", type=Path, default=Path("artifacts/traces"))
    parser.add_argument("--boxes", type=Path, help="Nose-box JSON when data is outside the repository root")
    args = parser.parse_args()
    extract(args.root, args.out, args.boxes)


if __name__ == "__main__":
    main()

```

### `thermal_face/windows.py`

```python
"""Shared timestamp-preserving windows and explicit evaluation exclusions."""

from dataclasses import dataclass
from pathlib import Path

import numpy as np

SAMPLE_HZ = 10.0
EXCLUDED_SUBJECTS = {"anestis": "unresolved frame/reference alignment"}


@dataclass
class Window:
    subject: str
    start_s: float
    end_s: float
    condition: str
    x: np.ndarray | None
    reference_bpm: float | None
    missing_fraction: float | None
    signal_rejection: str | None
    reference_rejection: str | None

    @property
    def eligible(self):
        return self.signal_rejection is None and self.reference_rejection is None

    def metadata(self):
        return {key: value for key, value in vars(self).items() if key != "x"} | {
            "eligible": self.eligible,
        }


def condition_for(start: float, end: float, boundary: float = 60.0) -> str:
    if end <= boundary:
        return "still"
    if start >= boundary:
        return "moving"
    return "transition"


def reference_rate(event_t, start, end):
    events = event_t[(event_t >= start) & (event_t < end)]
    if len(events) < 4:
        return None, "fewer_than_four_events"
    intervals = np.diff(events)
    if np.any((intervals < 0.8) | (intervals > 10.0)):
        return None, "implausible_event_interval"
    bpm = 60.0 / float(np.median(intervals))
    if not 6.0 <= bpm <= 45.0:
        return None, "reference_out_of_range"
    # Preserve the precision used by the existing trainers.
    return float(np.float32(bpm)), None


def resample_signal(t, trace, valid, target_t):
    good_t = t[valid]
    if len(good_t) < 2 or target_t[0] < good_t[0] or target_t[-1] > good_t[-1]:
        return None, None, "insufficient_valid_support"
    channels = np.stack([np.interp(target_t, good_t, trace[valid, i]) for i in range(9)])
    nearest = np.searchsorted(good_t, target_t).clip(1, len(good_t) - 1)
    gap = good_t[nearest] - good_t[nearest - 1]
    missing = (gap > 0.25).astype(np.float32)
    missing_fraction = float(missing.mean())
    if missing_fraction > 0.2:
        return None, missing_fraction, "missing_fraction_over_20_percent"
    channels -= np.median(channels, axis=1, keepdims=True)
    channels /= np.maximum(np.std(channels, axis=1, keepdims=True), 1.0)
    x = np.concatenate([channels, missing[None, :]], axis=0).astype(np.float32)
    return x, missing_fraction, None


def load_window_records(folder: Path, window_s=20.0, step_s=5.0, still_until_s=60.0):
    if not all(np.isfinite(v) and v > 0 for v in (window_s, step_s, still_until_s)):
        raise ValueError("Window, stride, and condition boundary must be positive and finite")
    grid = np.arange(0.0, window_s, 1.0 / SAMPLE_HZ, dtype=np.float64)
    records = {}
    for path in sorted(folder.glob("*.npz")):
        if path.stem.lower() in EXCLUDED_SUBJECTS:
            continue
        with np.load(path, allow_pickle=False) as data:
            t, trace, valid, marker = (data[k] for k in ("seconds", "trace", "valid", "marker"))
        if (t.ndim != 1 or len(t) < 2 or trace.shape != (len(t), 9)
                or valid.shape != t.shape or marker.shape != t.shape
                or valid.dtype != np.bool_ or not np.isfinite(t).all()
                or not np.isfinite(trace).all() or np.any(np.diff(t) <= 0)
                or not np.isin(marker, (0, 1, 2)).all()):
            raise ValueError(f"Invalid timestamped trace record: {path}")
        event_t = t[marker == 1]
        windows = []
        # Include a window ending exactly at the final timestamp.
        for start in np.arange(t[0], t[-1] - window_s + 1e-9, step_s):
            end = start + window_s
            bpm, reference_rejection = reference_rate(event_t, start, end)
            x, missing_fraction, signal_rejection = resample_signal(t, trace, valid, start + grid)
            windows.append(Window(
                path.stem, float(start), float(end),
                condition_for(start - t[0], end - t[0], still_until_s),
                x, bpm, missing_fraction, signal_rejection, reference_rejection,
            ))
        records[path.stem] = windows
    return records


def load_windows(folder: Path, window_s=20.0, step_s=5.0):
    """Compatibility view used by the CNN, GRU and spectral MLP trainers."""
    return {
        subject: [(w.x, np.float32(w.reference_bpm)) for w in windows if w.eligible]
        for subject, windows in load_window_records(folder, window_s, step_s).items()
    }

```

### `thermal_face/baselines.py`

```python
"""Spectral respiration baseline on traces resampled using actual timestamps."""

import numpy as np

from .windows import SAMPLE_HZ


def spectral_rate(x: np.ndarray) -> tuple[float | None, str | None]:
    """Average normalized nasal-patch power in the fixed 6--45 BPM band.

    Linear detrending and a Hann taper suppress drift and edge discontinuities.
    Eightfold zero padding samples the spectrum more densely; it does not
    improve the underlying resolution of a 20-second observation.
    """
    if x.ndim != 2 or x.shape[0] != 10 or x.shape[1] < 4 or not np.isfinite(x).all():
        raise ValueError("Expected finite 10-channel resampled traces")
    signal = x[:9].astype(np.float64)
    time = np.arange(signal.shape[1], dtype=np.float64) / SAMPLE_HZ
    time -= time.mean()
    signal -= signal.mean(axis=1, keepdims=True)
    signal -= ((signal @ time) / (time @ time))[:, None] * time
    active = np.std(signal, axis=1) > 1e-8
    if not active.any():
        return None, "no_spectral_energy"
    signal = signal[active] * np.hanning(signal.shape[1])
    n_fft = 8 * signal.shape[1]
    frequencies = np.fft.rfftfreq(n_fft, d=1.0 / SAMPLE_HZ)
    band = (frequencies >= 0.1) & (frequencies <= 0.75)
    power = np.abs(np.fft.rfft(signal, n=n_fft, axis=1))[:, band] ** 2
    total = power.sum(axis=1)
    active = total > 1e-16
    if not active.any():
        return None, "no_spectral_energy"
    spectrum = (power[active] / total[active, None]).mean(axis=0)
    return float(60.0 * frequencies[band][np.argmax(spectrum)]), None

```

### `thermal_face/train_mlp.py`

```python
"""Runnable small neural baseline on spectral features from nasal traces.

The four leave-one-person-out folds are exploratory because there is only one
session per person. The input extractor and all standardization are fit on
training people only, and each fold includes a train-median comparator.
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np

from .train_rate import load_windows


def spectral_features(trace: np.ndarray) -> np.ndarray:
    # 20 seconds at 10 Hz gives 3-breath/min raw FFT spacing.
    centered = trace[:9] - trace[:9].mean(axis=1, keepdims=True)
    taper = np.hanning(centered.shape[1])
    spectrum = np.abs(np.fft.rfft(centered * taper[None, :], axis=1))
    frequencies = np.fft.rfftfreq(centered.shape[1], d=0.1)
    band = (frequencies >= 0.1) & (frequencies <= 0.75)
    selected = spectrum[:, band]
    selected /= np.maximum(selected.sum(axis=1, keepdims=True), 1e-6)
    return np.concatenate([selected.ravel(), [float(trace[9].mean())]]).astype(np.float32)


def fit_predict(train_x: np.ndarray, train_y: np.ndarray, test_x: np.ndarray) -> np.ndarray:
    """Fit a regularized one-hidden-layer MLP using NumPy only."""
    x_mean = train_x.mean(axis=0)
    x_std = np.maximum(train_x.std(axis=0), 0.05)
    x = (train_x - x_mean) / x_std
    xt = (test_x - x_mean) / x_std
    y_mean = float(train_y.mean())
    y_std = max(float(train_y.std()), 1.0)
    y = ((train_y - y_mean) / y_std)[:, None]
    rng = np.random.default_rng(7)
    w1 = rng.normal(0, 0.06, size=(x.shape[1], 16))
    b1 = np.zeros((1, 16))
    w2 = rng.normal(0, 0.06, size=(16, 1))
    b2 = np.zeros((1, 1))
    params = [w1, b1, w2, b2]
    moments = [np.zeros_like(p) for p in params]
    velocities = [np.zeros_like(p) for p in params]
    for epoch in range(1, 801):
        hidden = np.tanh(x @ w1 + b1)
        prediction = hidden @ w2 + b2
        d_output = 2.0 * (prediction - y) / len(y)
        gradients = [
            x.T @ ((d_output @ w2.T) * (1.0 - hidden**2)) + 0.02 * w1,
            np.sum((d_output @ w2.T) * (1.0 - hidden**2), axis=0, keepdims=True),
            hidden.T @ d_output + 0.02 * w2,
            np.sum(d_output, axis=0, keepdims=True),
        ]
        for i, (param, grad) in enumerate(zip(params, gradients)):
            moments[i] = 0.9 * moments[i] + 0.1 * grad
            velocities[i] = 0.999 * velocities[i] + 0.001 * grad**2
            update = (moments[i] / (1.0 - 0.9**epoch)) / (
                np.sqrt(velocities[i] / (1.0 - 0.999**epoch)) + 1e-8
            )
            param -= 0.002 * update
    return ((np.tanh(xt @ w1 + b1) @ w2 + b2).ravel() * y_std + y_mean)


def run(traces: Path, out: Path) -> dict:
    examples = load_windows(traces)
    people = [person for person, samples in examples.items() if samples]
    if len(people) < 3:
        raise RuntimeError("Need at least three aligned subjects with valid windows")
    folds = []
    for held_out in people:
        train = [sample for person in people if person != held_out for sample in examples[person]]
        test = examples[held_out]
        train_x = np.stack([spectral_features(x) for x, _ in train])
        train_y = np.array([y for _, y in train])
        test_x = np.stack([spectral_features(x) for x, _ in test])
        test_y = np.array([y for _, y in test])
        predicted = fit_predict(train_x, train_y, test_x)
        baseline = float(np.median(train_y))
        folds.append({
            "held_out": held_out,
            "test_windows": len(test),
            "mlp_mae_bpm": round(float(np.mean(np.abs(predicted - test_y))), 3),
            "train_median_mae_bpm": round(float(np.mean(np.abs(baseline - test_y))), 3),
            "reference_mean_bpm": round(float(test_y.mean()), 3),
            "prediction_mean_bpm": round(float(predicted.mean()), 3),
        })
    report = {
        "status": "exploratory_four_people_one_session_each",
        "protocol": "leave-one-person-out; 20-second overlapping windows; no anestis due to alignment mismatch",
        "folds": folds,
        "macro_mlp_mae_bpm": round(float(np.mean([f["mlp_mae_bpm"] for f in folds])), 3),
        "macro_train_median_mae_bpm": round(float(np.mean([f["train_median_mae_bpm"] for f in folds])), 3),
    }
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(report, indent=2))
    return report


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--traces", type=Path, default=Path("artifacts/traces"))
    parser.add_argument("--out", type=Path, default=Path("artifacts/rate_mlp_pilot.json"))
    args = parser.parse_args()
    run(args.traces, args.out)


if __name__ == "__main__":
    main()

```

### `thermal_face/models.py`

```python
"""Small respiration regressors operating on nasal intensity traces."""

from torch import nn


class CNNRateNet(nn.Module):
    """Original temporal CNN retained as the comparison architecture."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(10, 24, 9, padding=4), nn.ReLU(), nn.AvgPool1d(2),
            nn.Conv1d(24, 32, 9, padding=4), nn.ReLU(), nn.AvgPool1d(2),
            nn.Conv1d(32, 32, 9, padding=4), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class CNNGRURateNet(nn.Module):
    """Convolutions extract local patterns; a GRU integrates their sequence.

    Input: (batch, 10, 200), nine nasal traces and a missing-data mask.
    Encoder: (batch, 32, 50). GRU: 50 steps with 32 features each.
    Output: one respiration rate per window, in breaths/minute.
    """

    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(10, 24, 9, padding=4), nn.ReLU(), nn.AvgPool1d(2),
            nn.Conv1d(24, 32, 9, padding=4), nn.ReLU(), nn.AvgPool1d(2),
        )
        self.rnn = nn.GRU(input_size=32, hidden_size=32, batch_first=True)
        self.head = nn.Sequential(nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1))

    def forward(self, x):
        sequence = self.encoder(x).transpose(1, 2)
        _, hidden = self.rnn(sequence)
        return self.head(hidden[-1]).squeeze(-1)

```

### `thermal_face/train_rate.py`

```python
"""CNN and CNN + GRU pilots with leave-one-person-out evaluation.

This is a research baseline, not a validated clinical or identity model. Train
only after auditing the labels and extracting traces. Each held-out person's
windows are entirely absent from that fold's training set.
"""

from __future__ import annotations

import argparse
import json
import random
import sys
from time import perf_counter
from pathlib import Path

from .windows import load_windows


def resolve_device(device="auto"):
    """An explicit CUDA request must never silently fall back to CPU."""
    import torch

    if device not in ("auto", "cpu", "cuda"):
        raise ValueError("device must be auto, cpu, or cuda")
    if device == "cpu":
        return torch.device("cpu")
    available = torch.cuda.is_available()
    if device == "cuda" and not available:
        raise RuntimeError("CUDA was requested but is unavailable. Install CUDA-enabled PyTorch and check the NVIDIA driver.")
    return torch.device("cuda", torch.cuda.current_device()) if available else torch.device("cpu")


def execution_environment(device="auto"):
    import platform
    import numpy as np
    import torch

    selected = resolve_device(device)
    return {
        "python": platform.python_version(), "numpy": np.__version__,
        "torch": torch.__version__, "requested_device": device,
        "device": str(selected),
        "device_name": torch.cuda.get_device_name(selected) if selected.type == "cuda" else "CPU",
        "cuda_runtime": torch.version.cuda,
    }


def fit_predict(train, test_x, epochs=40, seed=7, architecture="cnn-gru", device="auto", verbose=False):
    """Fit one fold using training labels only; predict unlabeled test inputs."""
    import numpy as np
    import torch
    from torch import nn
    from .models import CNNGRURateNet, CNNRateNet

    if epochs < 1 or architecture not in ("cnn", "cnn-gru"):
        raise ValueError("Positive epochs and a supported architecture are required")
    selected_device = resolve_device(device)
    started = perf_counter()
    if verbose:
        print(f"  {architecture}: initializing model and tensors on {selected_device}", flush=True)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(min(4, torch.get_num_threads()))
    model = (CNNGRURateNet if architecture == "cnn-gru" else CNNRateNet)().to(selected_device)
    train_x = torch.from_numpy(np.stack([x for x, _ in train])).to(selected_device)
    train_y = torch.from_numpy(np.array([y for _, y in train], dtype=np.float32)).to(selected_device)
    test_x = torch.from_numpy(np.asarray(test_x, dtype=np.float32)).to(selected_device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    for epoch in range(1, epochs + 1):
        show_progress = verbose and (epoch == 1 or epoch % 10 == 0 or epoch == epochs)
        if show_progress:
            epoch_loss = torch.zeros((), device=selected_device)
        model.train()
        # Keep CPU shuffle generation consistent across CPU and CUDA runs.
        order = torch.randperm(len(train_x)).to(selected_device)
        for ids in order.split(16):
            prediction = model(train_x[ids])
            loss = nn.functional.smooth_l1_loss(prediction, train_y[ids], beta=3.0)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if show_progress:
                epoch_loss += loss.detach() * len(ids)
        if show_progress:
            mean_loss = epoch_loss.item() / len(train_x)
            print(f"  {architecture}: epoch {epoch}/{epochs} | train loss {mean_loss:.4f} | "
                  f"{perf_counter() - started:.1f}s elapsed", flush=True)
    model.eval()
    with torch.no_grad():
        predicted = model(test_x).cpu().numpy()
    return predicted, sum(p.numel() for p in model.parameters() if p.requires_grad)


def run(folder: Path, out: Path, epochs: int = 40, seed: int = 7,
        architecture: str = "cnn-gru", device: str = "auto", verbose=False) -> dict:
    import numpy as np

    if epochs < 1:
        raise ValueError("epochs must be positive")
    if architecture not in ("cnn", "cnn-gru"):
        raise ValueError(f"Unknown architecture: {architecture}")
    environment = execution_environment(device)
    if verbose:
        print(f"Device: {environment['device']} ({environment['device_name']})", flush=True)
    metric_name = architecture.replace("-", "_")
    examples = load_windows(folder)
    people = [person for person, samples in examples.items() if samples]
    if len(people) < 3:
        raise RuntimeError("At least three aligned, nonempty subjects are needed")

    fold_results = []
    for fold_index, held_out in enumerate(people, 1):
        train = [sample for person in people if person != held_out for sample in examples[person]]
        test = examples[held_out]
        if verbose:
            print(f"Fold {fold_index}/{len(people)} | held out: {held_out} | "
                  f"train: {len(train)} windows | test: {len(test)} windows", flush=True)
        test_y = np.array([y for _, y in test])
        predicted, parameter_count = fit_predict(
            train, np.stack([x for x, _ in test]), epochs, seed, architecture, device, verbose=verbose,
        )
        baseline = float(np.median(np.array([y for _, y in train])))
        fold_results.append({
            "held_out": held_out,
            "train_subjects": [p for p in people if p != held_out],
            "test_windows": len(test),
            f"{metric_name}_mae_bpm": round(float(np.mean(np.abs(predicted - test_y))), 3),
            "train_median_baseline_mae_bpm": round(float(np.mean(np.abs(baseline - test_y))), 3),
            "mean_reference_bpm": round(float(np.mean(test_y)), 3),
            "mean_predicted_bpm": round(float(np.mean(predicted)), 3),
            "reference_bpm": test_y.tolist(),
            "predicted_bpm": predicted.tolist(),
        })
    report = {
        "status": "pilot_only_one_recording_per_person",
        "protocol": "leave-one-subject-out; 20s windows/5s stride; marker-1 median interval label",
        "architecture": architecture,
        "environment": environment,
        "epochs": epochs,
        "seed": seed,
        "trainable_parameters": parameter_count,
        "independent_test_people": len(people),
        "excluded_subjects": {"anestis": "unresolved frame/reference alignment"},
        "folds": fold_results,
        f"macro_{metric_name}_mae_bpm": round(float(np.mean([r[f"{metric_name}_mae_bpm"] for r in fold_results])), 3),
        "macro_baseline_mae_bpm": round(float(np.mean([r["train_median_baseline_mae_bpm"] for r in fold_results])), 3),
    }
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(report, indent=2))
    return report


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--traces", type=Path, default=Path("artifacts/traces"))
    parser.add_argument("--out", type=Path, help="Default: artifacts/rate_<architecture>_pilot.json")
    parser.add_argument("--architecture", choices=("cnn", "cnn-gru"), default="cnn-gru")
    parser.add_argument("--seed", type=int, default=7)
    parser.add_argument("--epochs", type=int, default=40)
    parser.add_argument("--device", choices=("auto", "cpu", "cuda"), default="auto",
                        help="auto selects CUDA when available; cuda requires a working GPU")
    args = parser.parse_args()
    model_name = args.architecture.replace("-", "_")
    out = args.out or Path(f"artifacts/rate_{model_name}_pilot.json")
    print("Starting training; initializing device and loading traces...", flush=True)
    try:
        run(args.traces, out, args.epochs, args.seed, args.architecture, args.device, verbose=True)
    except KeyboardInterrupt:
        print("\nTraining interrupted. Rerun the command to start again.", file=sys.stderr, flush=True)
        raise SystemExit(130)


if __name__ == "__main__":
    main()

```

### `thermal_face/evaluate.py`

```python
"""Compare respiration methods on common windows with explicit coverage."""

import argparse
from collections import Counter
import hashlib
import json
import sys
from pathlib import Path

import numpy as np

from .baselines import spectral_rate
from .train_rate import execution_environment, fit_predict
from .windows import EXCLUDED_SUBJECTS, SAMPLE_HZ, load_window_records

METHODS = ("train_median", "spectral", "cnn", "cnn_gru")
CONDITIONS = ("still", "moving", "transition")


def error_metrics(rows, method):
    errors = {}
    for row in rows:
        prediction = row["predictions_bpm"][method]
        if row["eligible"] and prediction is not None:
            errors.setdefault(row["subject"], []).append(abs(prediction - row["reference_bpm"]))
    per_subject = {p: float(np.mean(values)) for p, values in errors.items()}
    return {
        "evaluated_windows": sum(map(len, errors.values())),
        "evaluated_people": len(errors),
        "macro_mae_bpm": float(np.mean(list(per_subject.values()))) if errors else None,
        "pooled_window_mae_bpm": float(np.mean([e for values in errors.values() for e in values])) if errors else None,
        "per_subject_mae_bpm": per_subject,
    }


def summarize(rows):
    eligible = sum(row["eligible"] for row in rows)
    common = [row for row in rows if row["eligible"] and all(
        row["predictions_bpm"][method] is not None for method in METHODS
    )]
    methods = {}
    for method in METHODS:
        metrics = error_metrics(rows, method)
        predicted = metrics["evaluated_windows"]
        methods[method] = metrics | {
            "coverage_of_candidates": predicted / len(rows) if rows else None,
            "coverage_of_eligible": predicted / eligible if eligible else None,
            "abstention_reasons_on_eligible": dict(Counter(
                row["abstention_reasons"][method] for row in rows
                if row["eligible"] and row["predictions_bpm"][method] is None
            )),
        }
    return {
        "candidate_windows": len(rows),
        "signal_valid_windows": sum(row["signal_rejection"] is None for row in rows),
        "reference_valid_windows": sum(row["reference_rejection"] is None for row in rows),
        "eligible_windows": eligible,
        "eligibility_fraction": eligible / len(rows) if rows else None,
        "signal_rejections": dict(Counter(row["signal_rejection"] for row in rows if row["signal_rejection"])),
        "reference_rejections": dict(Counter(row["reference_rejection"] for row in rows if row["reference_rejection"])),
        "methods": methods,
        "common_comparison": {
            "windows": len(common),
            "methods": {method: error_metrics(common, method) for method in METHODS},
        },
    }


def run(traces: Path, out: Path, epochs=40, seed=7, still_until_s=60.0, device="auto", verbose=False):
    if epochs < 1:
        raise ValueError("epochs must be positive")
    if not traces.is_dir():
        raise FileNotFoundError(f"Trace directory does not exist: {traces}")
    environment = execution_environment(device)
    if verbose:
        print(f"Device: {environment['device']} ({environment['device_name']})", flush=True)
    records = load_window_records(traces, still_until_s=still_until_s)
    eligible = {subject: [w for w in windows if w.eligible] for subject, windows in records.items()}
    people = [subject for subject, windows in eligible.items() if windows]
    if len(people) < 3:
        raise RuntimeError("At least three nonempty eligible subjects are required")
    rows = []
    lookup = {}
    for subject, windows in records.items():
        for window in windows:
            row = window.metadata() | {
                "predictions_bpm": dict.fromkeys(METHODS),
                "abstention_reasons": dict.fromkeys(METHODS, None if window.eligible else "window_ineligible"),
            }
            rows.append(row)
            lookup[(subject, window.start_s)] = row
    folds = []
    for fold_index, held_out in enumerate(people, 1):
        training_people = [person for person in people if person != held_out]
        train = [(w.x, np.float32(w.reference_bpm)) for person in training_people for w in eligible[person]]
        test = eligible[held_out]
        test_x = np.stack([w.x for w in test])
        median = float(np.median(np.asarray([y for _, y in train], dtype=np.float32)))
        if verbose:
            print(f"Fold {fold_index}/{len(people)} | held out: {held_out} | "
                  f"train: {len(train)} windows | test: {len(test)} windows", flush=True)
        parameter_counts = {}
        for architecture in ("cnn", "cnn-gru"):
            predictions, parameter_counts[architecture] = fit_predict(train, test_x, epochs, seed, architecture, device, verbose=verbose)
            if np.shape(predictions) != (len(test),):
                raise ValueError(f"Unexpected predictions from {architecture}")
            for window, prediction in zip(test, predictions):
                row = lookup[(held_out, window.start_s)]
                method = architecture.replace("-", "_")
                if np.isfinite(prediction):
                    row["predictions_bpm"][method] = float(prediction)
                else:
                    row["abstention_reasons"][method] = "nonfinite_prediction"
        for window in test:
            row = lookup[(held_out, window.start_s)]
            row["predictions_bpm"]["train_median"] = median
            prediction, reason = spectral_rate(window.x)
            row["predictions_bpm"]["spectral"] = prediction
            row["abstention_reasons"]["spectral"] = reason
        folds.append({
            "held_out": held_out, "train_subjects": training_people,
            "train_windows": len(train), "test_windows": len(test),
            "train_median_bpm": median, "trainable_parameters": parameter_counts,
        })
    report = {
        "status": "exploratory_one_recording_per_person",
        "protocol": {
            "split": "leave-one-person-out", "window_s": 20.0, "step_s": 5.0,
            "sample_hz": SAMPLE_HZ, "epochs": epochs, "seed": seed,
            "still_until_s": still_until_s,
            "condition_labels": "assumed from elapsed recording time; boundary-crossing windows are transition",
            "reference": "60 / median marker-1 inter-event interval within each window; at least four events",
            "alignment": "assumed frame/CSV row correspondence; not independently verified",
            "coverage": "predictions / all complete candidate windows; reference screening is included, not deployment availability",
            "primary_comparison": "common_comparison uses only windows with predictions from every method",
            "spectral": "timestamp-resampled 10 Hz traces; linear detrend; Hann taper; mean normalized patch power; 6-45 BPM; 8x zero padding",
        },
        "environment": environment,
        "trace_sha256": {person: hashlib.sha256((traces / f"{person}.npz").read_bytes()).hexdigest() for person in records},
        "independent_test_people": len(people),
        "excluded_subjects": EXCLUDED_SUBJECTS,
        "subjects_without_eligible_windows": [p for p in records if not eligible[p]],
        "folds": folds,
        "overall": summarize(rows),
        "by_condition": {condition: summarize([row for row in rows if row["condition"] == condition]) for condition in CONDITIONS},
        "by_subject": {person: summarize([row for row in rows if row["subject"] == person]) for person in records},
        "windows": rows,
    }
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(report, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--traces", type=Path, default=Path("artifacts/traces"))
    parser.add_argument("--out", type=Path, default=Path("artifacts/evaluation.json"))
    parser.add_argument("--epochs", type=int, default=40)
    parser.add_argument("--seed", type=int, default=7)
    parser.add_argument("--device", choices=("auto", "cpu", "cuda"), default="auto",
                        help="auto selects CUDA when available; cuda requires a working GPU")
    parser.add_argument("--still-until-s", type=float, default=60.0,
                        help="Assumed still/moving boundary in elapsed seconds")
    args = parser.parse_args()
    print("Starting evaluation; initializing device and loading traces...", flush=True)
    try:
        report = run(args.traces, args.out, args.epochs, args.seed, args.still_until_s, args.device, verbose=True)
    except KeyboardInterrupt:
        print("\nEvaluation interrupted. Rerun the command to start again.", file=sys.stderr, flush=True)
        raise SystemExit(130)
    summary = report["overall"]
    print(f"Report: {args.out}")
    print(f"Eligible windows: {summary['eligible_windows']}/{summary['candidate_windows']}; "
          f"held-out people: {report['independent_test_people']}")
    for method, metrics in summary["common_comparison"]["methods"].items():
        mae = metrics["macro_mae_bpm"]
        rendered = f"{mae:.3f}" if mae is not None else "unavailable"
        print(f"{method}: common-window macro MAE {rendered} breaths/min")


if __name__ == "__main__":
    main()

```

## 14. Borrowing JoulesEye’s tracking idea: implementation and measured results

**Executed experiment · 17 October 2026**

We borrow [JoulesEye's](https://sustainability-lab.github.io/jouleseye/) tracking idea,
using [OpenCV CSRT](https://docs.opencv.org/4.6.0/d2/da2/classcv_1_1TrackerCSRT.html)
with the existing manually inspected first-frame boxes. RGB initialization is
unavailable in this dataset. This notebook presents actual saved experiment
outputs; reading those reports is executed below. Synthetic test outputs are
kept separate from real-data results. The source listings and opt-in rerun cells
make the extraction/training steps reviewable and reproducible.

The preceding sections document the fixed-crop baseline used in this comparison.
This local report embeds participant thermal frames.

### Fixed versus tracked nasal crops: experiment protocol

Specified before inspecting respiration errors for the tracked pipeline, 17 October 2026.

### Change being tested

Borrow JoulesEye's use of CSRT to follow the nasal region. Initialize from the
existing manually specified first-frame box in `config/nose_boxes.json` because
the supplied recordings contain no synchronized RGB. Use OpenCV 4.12.0 CSRT
defaults, including scale adaptation. Do not optimize tracker settings, crop
sizes, missing-data thresholds, or neural hyperparameters using test errors.
This is a thermal-only adaptation, not an exact JoulesEye reproduction.

Read original 8-bit frames and original CSV time. At each frame, compute the same
nine patch means from both the fixed and tracked box. Preserve every timestamp.
Skip CSRT updates on identical consecutive frames; retain the previous tracking
status and apply the existing >=5-frame frozen-run exclusion afterward. Mark
failed, nonfinite, undersized, or out-of-image boxes missing. Never substitute
the fixed crop after failure. Continue CSRT updates to allow recovery, without
claiming detector-based reinitialization. A successful tracker update is not
proof that the box still covers the nose.

Recompute fixed traces and require exact numerical equality to the earlier
`artifacts/traces` arrays. Write new outputs separately under
`artifacts/tracking_experiment/{fixed,tracked}`. Record per-frame boxes and
status, trace hashes, raw pixel hashes, source hashes, failures and runtime.

### Evaluation

- Keep the four previously eligible people; exclude `anestis` for unresolved alignment.
- Keep 20-second windows, five-second stride, 10 Hz resampling, marker-1 labels,
  normalization, missing threshold and 60-second condition boundary unchanged.
- Use the intersection of fixed/tracked eligible windows for **both training
  and testing**, separately for each person. Report each variant's original
  candidate coverage and exclusions; do not silently omit a person if all their
  paired windows are unavailable.
- Train both CNN and CNN + GRU from scratch for 40 epochs per held-out-person
  fold. Use the existing optimizer/loss/batch settings unchanged.
- Primary comparison: CNN + GRU, seed 7, macro MAE across held-out people.
- Predetermined sensitivity runs: seeds 17 and 27, for both architectures and crops.
- Include the training-median and spectral comparators. Score each method on
  matching windows with predictions from both crops, so spectral abstention
  cannot change the neural methods' scoring subset.
- Report overall, per-person, and still/moving/transition errors; per-seed paired
  changes; mean and population standard deviation over seeds. Negative
  tracked-minus-fixed MAE means improvement. Repeated seeds are not independent
  participants and do not provide population-level confidence intervals.
- Do not compare a coverage-reduced result directly with the old all-window
  2.986 BPM figure. Compare the retrained paired fixed baseline instead.

### References

- [JoulesEye project and method](https://sustainability-lab.github.io/jouleseye/)
- [OpenCV CSRT API](https://docs.opencv.org/4.6.0/d2/da2/classcv_1_1TrackerCSRT.html)

The main research question is whether this concrete adaptation improves this
pilot's measured error. It cannot establish reliable generalization from four
people or validate the anatomical accuracy of tracking without frame labels.

In [ ]:
from pathlib import Path
import json, sys, hashlib
import numpy as np
from IPython.display import display, Markdown, Image
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'thermal_face').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
FOLDER = ROOT / 'artifacts' / 'tracking_experiment'
report = json.loads((FOLDER / 'comparison.json').read_text())
audit = json.loads((FOLDER / 'tracking_audit.json').read_text())
print('Recorded training environment:', json.dumps(report['environment'], indent=2))
print('Tracker:', audit['tracker'], '| OpenCV:', audit['opencv'])
print('Seeds:', report['protocol']['seeds'], '| epochs:', report['protocol']['epochs'])
print('Training/evaluation wall seconds:', round(report['elapsed_seconds'], 2))
print('Extraction wall seconds:', round(sum(s['elapsed_seconds'] for s in audit['subjects'].values()), 2))
# Verify that the reports refer to the trace archives currently present.
for variant, hashes in report['trace_sha256'].items():
    for person, expected in hashes.items():
        assert hashlib.sha256((FOLDER / variant / f'{person}.npz').read_bytes()).hexdigest() == expected
print('All trace hashes verified.')

### Reproduce extraction and training (opt-in)
The saved results shown below were generated using these module entry points.
`RERUN=False` keeps a presentation from launching several minutes of tracking.
For this workspace only, OpenCV contrib was installed in
`artifacts/tracking_dependencies`; a clean environment can instead install
`requirements-tracking.txt`. Restart the notebook kernel after changing OpenCV
packages if an older cv2 module has already been imported.

In [ ]:
RERUN = False
if RERUN:
    local_cv = ROOT / 'artifacts' / 'tracking_dependencies'
    if local_cv.is_dir():
        sys.path.insert(0, str(local_cv))
    from thermal_face.tracking import extract
    from thermal_face.compare_tracking import run
    audit = extract(ROOT / 'Manuel_clean', FOLDER, ROOT / 'config' / 'nose_boxes.json',
                    ROOT / 'artifacts' / 'traces')
    report = run(FOLDER, FOLDER / 'comparison.json', seeds=[7, 17, 27], epochs=40, device='cpu')
else:
    print('Displaying the completed extraction and training reports; no retraining during this export.')

### Tracking behavior and missing-data accounting
Frozen runs keep their original time coordinates. Failed boxes would become
missing samples; they are never replaced by a fixed crop. Duplicate frames do
not update the tracker. The nine patch features, timestamp resampling and
window rules remain the same. CSRT includes scale adaptation, so patch areas
can change with the box dimensions. A zero failure count does not exclude drift.

In [ ]:
table = '| Person | Frames | Frozen | Failed tracking | Valid tracked | Fixed matches prior |\n|---|---:|---:|---:|---:|---|\n'
for person, s in audit['subjects'].items():
    table += f"| {person} | {s['frames']} | {s['frozen_frames']} | {s['tracking_failed_frames']} | {s['valid_tracked_frames']} | {s['fixed_matches_previous_trace_exactly']} |\n"
display(Markdown(table))
for variant, people in report['coverage'].items():
    print(variant, {p: {k: s[k] for k in ['candidates', 'eligible', 'paired']} for p, s in people.items()})

### Visual inspection: fixed box versus tracked box
Red = fixed starting box; green = CSRT box. Samples are selected at predetermined
elapsed times (0, 30, 60, 90, 115 seconds), not selected based on prediction error.
This qualitative inspection cannot replace annotated landmark accuracy.
Joao's 90 s and 115 s boxes visibly drift to the lower face/neck, while
Claudio's move toward the upper face/eye area. These updates are not flagged as
failures by CSRT. No settings were adjusted in response to these observations.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image as PILImage
import io
def show_plot(fig):
    stream = io.BytesIO()
    fig.savefig(stream, format='png', dpi=140, bbox_inches='tight')
    plt.close(fig)
    display(Image(data=stream.getvalue()))
boxes = json.loads((ROOT / 'config' / 'nose_boxes.json').read_text())
people = sorted(audit['subjects'])
fig, axes = plt.subplots(len(people), 5, figsize=(14, 10))
for row, person in enumerate(people):
    paths = sorted((ROOT / 'Manuel_clean' / person).glob('frame_*.png'))
    with np.load(FOLDER / 'tracked' / f'{person}.npz') as z:
        seconds, tracked_boxes, valid = z['seconds'], z['boxes'], z['track_ok']
    for col, time_s in enumerate([0, 30, 60, 90, 115]):
        i = int(np.argmin(abs(seconds-time_s)))
        with PILImage.open(paths[i]) as picture:
            axes[row, col].imshow(np.array(picture), cmap='gray')
        for box, color in [(boxes[person], 'red'), (tracked_boxes[i], 'lime')]:
            if box[0] >= 0:
                x, y, w, h = box
                axes[row, col].add_patch(Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=1.3))
        axes[row, col].set_title(f'{person} {seconds[i]:.1f}s', fontsize=9)
        axes[row, col].axis('off')
fig.suptitle('Fixed (red) and tracked (green) nasal crops; no label-based selection')
fig.tight_layout()
show_plot(fig)

### CSRT tracking experiment — measured results

Executed on the restored original recordings, 17 October 2026. This is a
thermal-only adaptation of JoulesEye's tracking idea: the existing first-frame
boxes initialize CSRT; there is no RGB detector or detector-based recovery.
The complete [executed experiment notebook](tracking_experiment.ipynb) and
[HTML presentation](tracking_experiment.html) include code, per-frame diagnostics,
sample overlays, paired evaluation and limitations.

**Primary result (CNN + GRU, seed 7):** fixed crops **2.986 BPM** macro MAE;
tracked crops **2.969 BPM**. Tracking reduced
error by **0.017 BPM** (0.6%).

| Method | Fixed MAE | Tracked MAE | Tracked − fixed |
|---|---:|---:|---:|
| train_median | 2.501 | 2.501 | +0.000 |
| spectral | 10.618 | 10.809 | +0.190 |
| cnn | 3.577 | 3.268 | -0.309 |
| cnn_gru | 2.986 | 2.969 | -0.017 |

#### Predetermined seed sensitivity

| Seed | Fixed CNN + GRU | Tracked CNN + GRU | Tracked − fixed |
|---|---:|---:|---:|
| 7 | 2.986 | 2.969 | -0.017 |
| 17 | 14.867 | 14.865 | -0.002 |
| 27 | 2.557 | 2.551 | -0.006 |


Across the three predetermined seeds (7, 17, 27), CNN + GRU mean MAE is
**6.803 BPM fixed** versus
**6.795 BPM tracked**.
Tracking improves 3 of 3 seeds.
These seeds measure optimization sensitivity, not additional independent people.

The primary CNN + GRU gain is very small. Seed 17 produces much larger errors
for both crop variants than seeds 7 and 27; this indicates substantial training
sensitivity under the unchanged 40-epoch recipe. The three-seed mean must not be
presented without that context. Tracking does not solve this optimization issue.

For the CNN-only comparison, three-seed mean MAE changes from
3.682 to
3.247 BPM, with improvement in
3/3 seeds. This is a larger
numerical benefit, but the tracked CNN still does not beat the median baseline
on the three-seed mean. The spectral comparator becomes worse with tracking.

#### Coverage and fairness

- Fixed eligible windows: 77/84; tracked: 77/84.
- Both training and testing use the same 77 paired eligible windows,
  with leave-one-person-out splits and unchanged architectures, loss, optimizer,
  epoch count, normalization and reference labels.
- All four recomputed fixed traces matched the prior trace arrays exactly.
- `anestis` remains excluded because its frame/reference alignment is unresolved.
- Explicit CSRT failures: 0 frames.
  Successful updates are not a measurement of anatomical tracking accuracy.

#### Visual finding: undetected drift

Inspection of predetermined 90 s and 115 s samples shows Joao's tracked box
moving onto the lower face/neck rather than remaining on the nose. Claudio's
90 s and 115 s samples shift toward the upper face/eye area. CSRT still returns
successful updates on these frames. Thus a lower respiration error, if present,
does not by itself validate nasal tracking or prove that the learned signal is
physiological breathing. The notebook includes these overlays for review.

#### Interpretation and limitations

This experiment supports a reduction
in the primary model's seed-7 error on this pilot. It is not evidence that tracking
will generalize to new recordings. Only four people, one recording each, were
evaluated. Fixed initial boxes, possible undetected drift, uncertain marker
semantics, and the assumed motion boundary remain limitations. CSRT changes both
position and scale, so this experiment evaluates that combined change.

The training-median comparator remains essential: its paired seed-7 MAE is
2.501 BPM;
the tracked CNN + GRU does not beat it.
No tracker or model settings were revised after inspecting these errors.

Next, annotate nose locations through motion to quantify drift, test localization
and explicit recovery, verify reference events/alignment, and collect additional
people and sessions. Do not replace a baseline based solely on a favorable seed.

#### Reproduction

Install `requirements-tracking.txt` in a clean environment (use the contrib
OpenCV distribution rather than combining multiple OpenCV wheels), then run:

```powershell
python -m thermal_face.tracking --root Manuel_clean --boxes config/nose_boxes.json --out artifacts/tracking_experiment --fixed-reference artifacts/traces
python -m thermal_face.compare_tracking --traces artifacts/tracking_experiment --epochs 40 --seeds 7 17 27 --device cpu
python docs/build_tracking_report.py
```

Machine-readable reports: `artifacts/tracking_experiment/tracking_audit.json`
and `artifacts/tracking_experiment/comparison.json`. The protocol is documented
in [tracking_experiment_protocol.md](tracking_experiment_protocol.md).

### Per-person and condition results, primary seed 7
All errors are breaths/minute. Each person contributes equally to macro MAE.
The four people are the independent evaluation units; overlapping windows and
repeated seeds do not enlarge that number.

In [ ]:
primary = report['runs'][0]
table = '| Person | Fixed CNN + GRU | Tracked CNN + GRU | Change |\n|---|---:|---:|---:|\n'
for person in sorted(primary['comparison']['cnn_gru']['fixed']['per_subject_mae_bpm']):
    a, b = [primary['comparison']['cnn_gru'][v]['per_subject_mae_bpm'][person] for v in ['fixed', 'tracked']]
    table += f'| {person} | {a:.3f} | {b:.3f} | {b-a:+.3f} |\n'
display(Markdown(table))
table = '| Condition | Fixed CNN + GRU | Tracked CNN + GRU |\n|---|---:|---:|\n'
for condition in ['still', 'moving', 'transition']:
    vals = [primary['variants'][v]['by_condition'][condition]['methods']['cnn_gru']['macro_mae_bpm'] for v in ['fixed', 'tracked']]
    table += '| ' + condition + ' | ' + ' | '.join(f'{v:.3f}' if v is not None else 'N/A' for v in vals) + ' |\n'
display(Markdown(table))
fig, ax = plt.subplots(figsize=(9, 4))
methods = list(primary['comparison'])
positions = np.arange(len(methods))
for offset, variant, color in [(-0.18, 'fixed', '#637c91'), (0.18, 'tracked', '#188c7b')]:
    vals = [primary['comparison'][m][variant]['macro_mae_bpm'] for m in methods]
    bars = ax.bar(positions+offset, vals, 0.36, label=variant, color=color)
    ax.bar_label(bars, fmt='%.2f', padding=3)
ax.set(xticks=positions, xticklabels=methods, ylabel='Macro MAE (BPM)', title='Measured paired comparison: seed 7')
ax.legend()
show_plot(fig)

### Verify implementation checks
This output is the saved test log from the implementation run. It includes
synthetic translated-target tracking, failure/recovery handling, frozen-run
timing, matched train/test windows, and existing evaluation/model tests.
These checks do not establish anatomical accuracy or clinical validity.

In [ ]:
text = (ROOT / 'artifacts' / 'tracking_tests.txt').read_text(encoding='utf-8')
lines = text.splitlines()
print('\n'.join(line for line in lines if line.startswith('test_')))
print('\n'.join(lines[-5:]))

### Implementation source used for this experiment
Reference listings; rerun cells above call these modules.

#### `thermal_face/tracking.py`

```python
"""CSRT nasal tracking experiment, initialized from the existing first-frame box.

No RGB initialization, detector, label-based tuning, or manual recovery is used.
Unsuccessful/out-of-frame boxes are missing measurements, never fixed-box fallback.
"""
from __future__ import annotations

import argparse
import hashlib
import json
from pathlib import Path
from time import perf_counter

import numpy as np

from .data import SUBJECTS, read_respiration


def create_tracker():
    import cv2
    factory = getattr(cv2, 'TrackerCSRT_create', None)
    if factory is None:
        factory = getattr(getattr(cv2, 'legacy', None), 'TrackerCSRT_create', None)
    if factory is None:
        raise RuntimeError('CSRT requires opencv-contrib-python; the base opencv-python package is insufficient')
    return factory()  # Published OpenCV defaults; no tuning against respiration labels.


def integer_box(box, shape):
    """Reject invalid boxes rather than silently changing the region by clipping."""
    values = np.asarray(box, dtype=float)
    if values.shape != (4,) or not np.isfinite(values).all():
        return None
    x, y, w, h = (int(round(v)) for v in values)
    if min(w, h) < 3 or min(x, y) < 0 or x + w > shape[1] or y + h > shape[0]:
        return None
    return x, y, w, h


def patch_means(frame, box):
    x, y, w, h = box
    crop = frame[y:y+h, x:x+w].astype(np.float32)
    return np.asarray([crop[r*h//3:(r+1)*h//3, c*w//3:(c+1)*w//3].mean()
                       for r in range(3) for c in range(3)], dtype=np.float32)


def unfrozen_mask(repeated):
    """Match the fixed extractor's >=5 identical-frame exclusion exactly."""
    valid = np.ones(len(repeated), dtype=bool)
    start = None
    for i in range(1, len(repeated) + 1):
        same = i < len(repeated) and repeated[i]
        if same and start is None:
            start = i - 1
        elif not same and start is not None:
            if i - start >= 5:
                valid[start:i] = False
            start = None
    return valid


class NoseTracker:
    """Track only images; no access to subject labels or reference events."""
    def __init__(self, frame, box, factory=create_tracker):
        self.tracker = factory()
        self.box = integer_box(box, frame.shape)
        if self.box is None:
            raise ValueError('Initial box must fit inside the image and contain nine nonempty patches')
        result = self.tracker.init(frame, self.box)
        if result is False:
            raise RuntimeError('CSRT initialization failed')
        self.ok = True

    def update(self, frame, repeated=False):
        if repeated:
            return self.box if self.ok else None, 'duplicate' if self.ok else 'lost'
        was_ok = self.ok
        success, candidate = self.tracker.update(frame)
        box = integer_box(candidate, frame.shape) if success else None
        self.ok = box is not None
        if self.ok:
            self.box = box
        # Continue asking CSRT to update after loss; no synthetic/reinitialized boxes.
        return box, ('tracked' if was_ok else 'recovered') if self.ok else 'lost'


def extract(root: Path, out: Path, boxes_path: Path, fixed_reference: Path | None = None,
            subjects=SUBJECTS):
    import cv2
    cv2.setNumThreads(2)
    cv2.setRNGSeed(7)
    boxes = json.loads(boxes_path.read_text(encoding='utf-8'))
    out.mkdir(parents=True, exist_ok=True)
    for variant in ['fixed', 'tracked']:
        (out / variant).mkdir(exist_ok=True)
    report = {
        'tracker': 'OpenCV CSRT, default parameters', 'opencv': cv2.__version__,
        'initialization': 'existing hand-inspected first-frame box; no RGB',
        'failure_policy': 'missing; no fixed-box fallback; subsequent CSRT updates may recover',
        'freeze_policy': 'skip tracker update on identical frames; flag runs >=5 as missing',
        'box_policy': 'CSRT position and scale; rounded integer boxes; reject outside image',
        'seed': 7, 'opencv_threads': 2, 'subjects': {}, 'excluded': {},
        'boxes_sha256': hashlib.sha256(boxes_path.read_bytes()).hexdigest(),
        'source_sha256': hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
    }
    for subject in subjects:
        paths = sorted((root / subject).glob('frame_*.png'))
        times, markers = read_respiration(next(root.glob(f'*{subject}_respiration.csv')))
        if len(paths) != len(times):
            report['excluded'][subject] = f'{len(paths)} frames versus {len(times)} reference rows'
            print(f'Skipping {subject}: unverified alignment', flush=True)
            continue
        if not paths:
            raise ValueError(f'No frames for {subject}')
        started = perf_counter()
        fixed = np.zeros((len(paths), 9), dtype=np.float32)
        tracked = np.zeros_like(fixed)
        repeated = np.zeros(len(paths), dtype=bool)
        track_ok = np.zeros(len(paths), dtype=bool)
        track_boxes = np.full((len(paths), 4), -1, dtype=np.int32)
        states = np.empty(len(paths), dtype='U10')
        previous = None
        pixels_hash = hashlib.sha256()
        print(f'{subject}: extracting and tracking {len(paths)} original frames', flush=True)
        for i, path in enumerate(paths):
            frame = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
            if frame is None:
                raise ValueError(f'Unreadable image: {path}')
            if frame.ndim == 3 and frame.shape[2] == 3:
                if not (np.array_equal(frame[:, :, 0], frame[:, :, 1]) and
                        np.array_equal(frame[:, :, 0], frame[:, :, 2])):
                    raise ValueError(f'Expected identical grayscale channels: {path}')
                gray = frame[:, :, 0]
            elif frame.ndim == 2:
                gray = frame
                frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
            else:
                raise ValueError(f'Unsupported frame: {path}')
            if gray.dtype != np.uint8:
                raise ValueError('This experiment expects the original 8-bit images')
            pixels_hash.update(gray.tobytes())
            fixed_box = integer_box(boxes[subject], gray.shape)
            if fixed_box is None:
                raise ValueError(f'Invalid configured box for {subject}')
            fixed[i] = patch_means(gray, fixed_box)
            if i == 0:
                tracker = NoseTracker(frame, fixed_box)
                box, state = fixed_box, 'initial'
            else:
                repeated[i] = np.array_equal(gray, previous)
                box, state = tracker.update(frame, bool(repeated[i]))
            previous = gray.copy()
            states[i] = state
            if box is not None:
                track_ok[i] = True
                track_boxes[i] = box
                tracked[i] = patch_means(gray, box)
            if (i + 1) % 500 == 0:
                print(f'  {subject}: {i+1}/{len(paths)} frames; {np.count_nonzero(~track_ok[:i+1])} failed boxes; '
                      f'{perf_counter()-started:.1f}s', flush=True)
        valid = unfrozen_mask(repeated)
        seconds = np.array([(t-times[0]).total_seconds() for t in times])
        base = dict(seconds=seconds, marker=np.asarray(markers, dtype=np.int8))
        comparison = None
        if fixed_reference is not None:
            reference = fixed_reference / f'{subject}.npz'
            with np.load(reference, allow_pickle=False) as saved:
                comparison = all(np.array_equal(saved[k], v) for k, v in
                                 (base | {'trace': fixed, 'valid': valid}).items())
            if not comparison:
                raise AssertionError(f'Fixed extraction differs from the existing baseline: {subject}')
        np.savez_compressed(out / 'fixed' / f'{subject}.npz', **base, trace=fixed, valid=valid)
        np.savez_compressed(out / 'tracked' / f'{subject}.npz', **base,
                            trace=tracked, valid=valid & track_ok,
                            track_ok=track_ok, boxes=track_boxes, state=states, frozen=~valid)
        info = {
            'frames': len(paths), 'frozen_frames': int((~valid).sum()),
            'tracking_failed_frames': int((~track_ok).sum()),
            'tracking_success_fraction': float(track_ok.mean()),
            'valid_tracked_frames': int((valid & track_ok).sum()),
            'recoveries': int((states == 'recovered').sum()),
            'source_pixel_sha256': pixels_hash.hexdigest(),
            'reference_csv_sha256': hashlib.sha256(next(root.glob(f'*{subject}_respiration.csv')).read_bytes()).hexdigest(),
            'fixed_matches_previous_trace_exactly': comparison,
            'elapsed_seconds': perf_counter()-started,
        }
        report['subjects'][subject] = info
        (out / 'tracking_audit.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
        print(f'{subject} completed: {info}', flush=True)
    (out / 'tracking_audit.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--root', type=Path, required=True)
    parser.add_argument('--out', type=Path, default=Path('artifacts/tracking_experiment'))
    parser.add_argument('--boxes', type=Path, default=Path('config/nose_boxes.json'))
    parser.add_argument('--fixed-reference', type=Path)
    args = parser.parse_args()
    extract(args.root, args.out, args.boxes, args.fixed_reference)


if __name__ == '__main__':
    main()

```

#### `thermal_face/compare_tracking.py`

```python
"""Paired fixed-versus-CSRT evaluation on identical training and test windows."""
from __future__ import annotations

import argparse
import hashlib
import json
from pathlib import Path
from time import perf_counter

import numpy as np

from .baselines import spectral_rate
from .evaluate import METHODS, summarize
from .train_rate import execution_environment, fit_predict
from .windows import load_window_records


def paired_windows(fixed, tracked):
    """Use the eligibility intersection for BOTH training and testing."""
    if set(fixed) != set(tracked) or len(fixed) < 3:
        raise ValueError('Need the same three or more subjects in both variants')
    paired = {}
    for subject in sorted(fixed):
        a = {w.start_s: w for w in fixed[subject]}
        b = {w.start_s: w for w in tracked[subject]}
        if a.keys() != b.keys():
            raise ValueError(f'Candidate timestamps differ for {subject}')
        for start in a:
            wa, wb = a[start], b[start]
            if (wa.end_s, wa.condition, wa.reference_bpm, wa.reference_rejection) != (
                    wb.end_s, wb.condition, wb.reference_bpm, wb.reference_rejection):
                raise ValueError(f'Reference labels or conditions differ for {subject} at {start}')
        paired[subject] = [start for start in sorted(a) if a[start].eligible and b[start].eligible]
        if not paired[subject]:
            raise ValueError(f'No paired eligible windows for {subject}; cannot silently drop a person')
    return paired


def run(folder: Path, out: Path, seeds=(7, 17, 27), epochs=40, device='cpu'):
    seeds = tuple(seeds)
    if not seeds or len(set(seeds)) != len(seeds) or epochs < 1:
        raise ValueError('Specify distinct seeds and positive epochs')
    started = perf_counter()
    records = {variant: load_window_records(folder / variant) for variant in ['fixed', 'tracked']}
    paired = paired_windows(records['fixed'], records['tracked'])
    people = sorted(paired)
    coverage = {}
    for variant in records:
        coverage[variant] = {}
        for person, windows in records[variant].items():
            coverage[variant][person] = {
                'candidates': len(windows), 'eligible': sum(w.eligible for w in windows),
                'paired': len(paired[person]),
                'windows': [w.metadata() for w in windows],
            }
    report = {
        'protocol': {
            'primary_seed': seeds[0], 'seeds': list(seeds), 'epochs': epochs,
            'split': 'leave-one-person-out',
            'paired_training_and_test': True,
            'paired_rule': 'intersection of fixed/tracked eligibility by subject and exact window timestamp',
            'window_s': 20, 'step_s': 5, 'sample_hz': 10,
            'model_selection': 'no selection or tuning using held-out results',
            'interpretation': 'seed variation is not independent-person uncertainty; four-person exploratory pilot',
            'excluded_subjects': {'anestis': 'unresolved frame/reference alignment'},
        },
        'environment': execution_environment(device), 'coverage': coverage,
        'paired_starts': paired, 'runs': [],
        'trace_sha256': {variant: {person: hashlib.sha256((folder / variant / f'{person}.npz').read_bytes()).hexdigest()
                                    for person in people} for variant in records},
        'source_sha256': {name: hashlib.sha256(Path(__file__).with_name(name).read_bytes()).hexdigest()
                          for name in ['compare_tracking.py', 'train_rate.py', 'models.py', 'windows.py', 'baselines.py']},
    }
    for seed in seeds:
        seed_result = {'seed': seed, 'variants': {}}
        for variant in records:
            examples = {person: [w for w in records[variant][person] if w.start_s in paired[person]] for person in people}
            rows, folds = [], []
            for held_out in people:
                training_people = [p for p in people if p != held_out]
                train = [(w.x, np.float32(w.reference_bpm)) for p in training_people for w in examples[p]]
                test = examples[held_out]
                test_x = np.stack([w.x for w in test])
                median = float(np.median(np.array([y for _, y in train], dtype=np.float32)))
                print(f'seed={seed} {variant} held_out={held_out}: train={len(train)}, test={len(test)}', flush=True)
                predicted = {}
                parameters = {}
                for architecture in ['cnn', 'cnn-gru']:
                    values, count = fit_predict(train, test_x, epochs, seed, architecture, device)
                    if np.shape(values) != (len(test),) or not np.isfinite(values).all():
                        raise ValueError('Nonfinite or malformed neural predictions; do not silently exclude them')
                    predicted[architecture.replace('-', '_')] = values
                    parameters[architecture] = count
                for i, window in enumerate(test):
                    spectrum, reason = spectral_rate(window.x)
                    row = window.metadata() | {
                        'predictions_bpm': {'train_median': median, 'spectral': spectrum,
                                            'cnn': float(predicted['cnn'][i]),
                                            'cnn_gru': float(predicted['cnn_gru'][i])},
                        'abstention_reasons': dict.fromkeys(METHODS),
                    }
                    row['abstention_reasons']['spectral'] = reason
                    rows.append(row)
                folds.append({'held_out': held_out, 'train_subjects': training_people,
                              'train_windows': len(train), 'test_windows': len(test),
                              'train_median_bpm': median, 'parameters': parameters})
            seed_result['variants'][variant] = {
                'folds': folds, 'windows': rows, 'overall': summarize(rows),
                'by_condition': {c: summarize([r for r in rows if r['condition'] == c]) for c in ['still', 'moving', 'transition']},
                'by_subject': {p: summarize([r for r in rows if r['subject'] == p]) for p in people},
            }
        # Pair predictions separately per method, so spectral abstention cannot affect CNN scoring.
        comparisons = {}
        variants = seed_result['variants']
        index = {v: {(r['subject'], r['start_s']): r for r in variants[v]['windows']} for v in variants}
        for method in METHODS:
            keys = [k for k in index['fixed'] if all(index[v][k]['predictions_bpm'][method] is not None for v in index)]
            from .evaluate import error_metrics
            metrics = {v: error_metrics([index[v][k] for k in keys], method) for v in index}
            a, b = [metrics[v]['macro_mae_bpm'] for v in ['fixed', 'tracked']]
            comparisons[method] = {
                **metrics, 'paired_prediction_windows': len(keys),
                'tracked_minus_fixed_mae_bpm': b-a if a is not None and b is not None else None,
            }
        seed_result['comparison'] = comparisons
        report['runs'].append(seed_result)
        print('Seed comparison:', json.dumps(comparisons, indent=2), flush=True)
    report['aggregate'] = {}
    for method in METHODS:
        values = np.array([[r['comparison'][method][v]['macro_mae_bpm'] for v in ['fixed', 'tracked']]
                           for r in report['runs']], dtype=float)
        if not np.isfinite(values).all():
            report['aggregate'][method] = {'status': 'no paired predictions in at least one seed'}
            continue
        differences = values[:, 1] - values[:, 0]
        report['aggregate'][method] = {
            'fixed_mean_mae_bpm': float(values[:, 0].mean()),
            'tracked_mean_mae_bpm': float(values[:, 1].mean()),
            'fixed_seed_sd_bpm': float(values[:, 0].std()),
            'tracked_seed_sd_bpm': float(values[:, 1].std()),
            'mean_tracked_minus_fixed_bpm': float(differences.mean()),
            'seeds_with_improvement': int((differences < 0).sum()),
        }
    report['elapsed_seconds'] = perf_counter()-started
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(report, indent=2, allow_nan=False), encoding='utf-8')
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--traces', type=Path, default=Path('artifacts/tracking_experiment'))
    parser.add_argument('--out', type=Path, default=Path('artifacts/tracking_experiment/comparison.json'))
    parser.add_argument('--epochs', type=int, default=40)
    parser.add_argument('--seeds', type=int, nargs='+', default=[7, 17, 27])
    parser.add_argument('--device', choices=['cpu', 'cuda', 'auto'], default='cpu')
    args = parser.parse_args()
    report = run(args.traces, args.out, args.seeds, args.epochs, args.device)
    print(json.dumps(report['aggregate'], indent=2))


if __name__ == '__main__':
    main()

```

## 15. Kwon BAFR reproduction: corrected implementation and coverage experiment
**Executed 18 October 2026.** This is the Kwon-inspired segmentation branch of the broader breathing-rate detector project. Earlier CNN/GRU and CSRT studies test other parts of the same project.

Source: Kwon et al., *Breathing-Associated Facial Region Segmentation for Thermal Camera-Based Indirect Breathing Monitoring*, [DOI 10.1109/JTEHM.2023.3295775](https://doi.org/10.1109/JTEHM.2023.3295775), supplied PDF pages 3–5, equations 1–2 and Figure 4.

The paper specifies KLT nasal tracking → four-second BAF → maximum-normalized BAP → two-class MRF with 30 ICM iterations per frame → integrated frame differences. **BAF is sample standard deviation, not variance alone.** Our earlier criticism of the square root was incorrect and is withdrawn.

This chapter records a method reimplementation on different data, not a reproduction of the paper's published accuracy. The complete assumptions were recorded before evaluating errors in `docs/kwon_reproduction_protocol.md`.

In [ ]:
from pathlib import Path
import json, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
ROOT = Path.cwd()
if not (ROOT / 'thermal_face').exists():
    ROOT = ROOT.parent
folder = ROOT / 'artifacts/kwon_experiment'
evaluation = json.loads((folder / 'evaluation.json').read_text())
audit = json.loads((folder / 'audit.json').read_text())
print('Recorded status:', evaluation['status'])
print('Primary contrast:', evaluation['primary'])
display(Markdown((ROOT / 'docs/kwon_reproduction_protocol.md').read_text()))

## Step 1. Correct the implementation before judging segmentation
The legacy code estimated reference-to-current motion but applied it in the opposite sampling direction. The new code uses `WARP_INVERSE_MAP` with the correct ROI-coordinate offset. It computes a fresh four-second sample-SD map and MRF mask at each valid frame, instead of one whole-recording mask. Original timestamps and exclusions remain intact.

The signal integrates the mean frame difference under the current mask. Changing mask membership alone cannot create an intensity jump. Gaussian likelihoods, four-neighbor Potts weight 0.6, red-black ICM, and KLT thresholds are explicitly declared choices because the paper does not fully specify them. These are not claimed to be the authors' original implementation.

The following cell exposes the actual extraction and evaluation source for presentation and review.

In [ ]:
display(Markdown('```python\n' + (ROOT / 'thermal_face/kwon.py').read_text() + '\n```'))
display(Markdown('```python\n' + (ROOT / 'thermal_face/evaluate_kwon.py').read_text() + '\n```'))

## Step 2. Run four controlled variants
The experiment used original frames for Joao, Claudio, Manuel and Jaime. `anestis` remains excluded because frame and CSV counts disagree.

1. Fixed nasal ROI, no segmentation.
2. Corrected KLT alignment, no segmentation.
3. Corrected KLT plus dynamic BAF/BAP/MRF.
4. Deliberately wrong warp plus the same dynamic MRF, isolating warp direction.

Commands actually run:
```powershell
python -m thermal_face.kwon --root Manuel_clean --out artifacts/kwon_experiment --boxes config/nose_boxes.json
python -m thermal_face.evaluate_kwon --traces artifacts/kwon_experiment --out artifacts/kwon_experiment/evaluation.json
python -m unittest discover -s tests -v
```
The report cells read saved experiment outputs; they do not silently reprocess data. Source and input hashes are recorded in the audit and evaluation files.

In [ ]:
print((ROOT / 'artifacts/kwon_tests.txt').read_text().split('----------------------------------------------------------------------')[-1])
for person, info in audit['subjects'].items():
    print(person, json.dumps(info, indent=2))

## Step 3. Check coverage before interpreting rate errors
The unchanged evaluator uses 20-second windows every five seconds. It rejects windows with more than 20% missing interpolated support, and does not extrapolate beyond valid endpoints. The four-second BAF support rule additionally rejects windows with fewer than 80% valid source samples or less than 3.5 seconds of valid span.

Frozen frames, tracking failures and BAF warm-up/recovery compound. This is a pipeline coverage failure under the declared rules; it is not evidence that the MRF cannot identify breathing pixels.

In [ ]:
rows = ['| Variant | Eligible / candidate windows | Signal exclusions |', '|---|---:|---|']
for name, result in evaluation['variants'].items():
    rows.append(f"| {name} | {result['eligible']} / {result['candidates']} | {result['signal_exclusions']} |")
display(Markdown('\n'.join(rows)))
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
for ax, person in zip(axes, ['Joao', 'Claudio', 'Manuel', 'Jaime']):
    for level, variant in enumerate(evaluation['variants']):
        z = np.load(folder / variant / f'{person}.npz')
        ax.scatter(z['seconds'][z['valid']], np.full(z['valid'].sum(), level), s=1)
    ax.set_yticks(range(4), list(evaluation['variants']))
    ax.set_title(person); ax.grid(alpha=.2)
axes[-1].set_xlabel('Original elapsed seconds; dots indicate valid extracted samples')
fig.tight_layout(); fig.savefig(folder / 'coverage.png', dpi=150); display(Image(filename=str(folder / 'coverage.png'))); plt.close(fig)

## Step 4. Inspect fixed-time segmentation examples
These snapshots were selected at fixed times before error evaluation. Black masks can represent unavailable BAF support, not necessarily a failed MRF optimization. Masks label high thermal variability; without pixel annotations we cannot establish anatomical accuracy or Dice/IoU. Visual inspection shows cheek and facial-edge regions in several masks, alongside nasal regions. Spatial smoothness alone does not establish breathing specificity.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for row, person in enumerate(['Joao', 'Claudio', 'Manuel', 'Jaime']):
    z = np.load(folder / 'diagnostics' / f'{person}.npz')
    for col, (key, label) in enumerate([('raw_samples','Raw crop'), ('aligned_samples','Correct KLT'), ('klt_mrf_bap','BAP'), ('klt_mrf_masks','MRF')]):
        axes[row, col].imshow(z[key][0], cmap='gray' if col < 2 else 'magma')
        axes[row, col].set_title(f"{person}, {z['sample_seconds'][0]:.1f}s: {label}", fontsize=9)
        axes[row, col].axis('off')
fig.tight_layout(); fig.savefig(folder / 'masks.png', dpi=150); display(Image(filename=str(folder / 'masks.png'))); plt.close(fig)

## Step 5. Report the measured result without substituting another endpoint
**The primary MRF comparison is unevaluable: 0 common eligible windows, including 0/36 still candidates. Its MAE is undefined, not zero.** No improvement claim is supported. Both correct- and wrong-warp MRF branches have zero eligible windows, so real-data rate errors cannot quantify the benefit of correcting warp direction.

The auxiliary alignment-only comparison is evaluable on identical windows. Peak-count and spectral outcomes differ; neither is a substitute for the primary segmentation result.

In [ ]:
rows = ['| Contrast / estimator / condition | Paired windows | First MAE | Second MAE | Change (BPM) |', '|---|---:|---:|---:|---:|']
fmt = lambda value: 'undefined' if value is None else f'{value:.3f}'
for name in ['segmentation', 'stabilization', 'warp_correction']:
    for method in ['spectral', 'peak_count']:
        for condition in ['still', 'overall']:
            r = evaluation['comparisons'][name][method][condition]
            rows.append(f"| {name} / {method} / {condition} | {r['common_windows']} | {fmt(r['first']['macro_mae_bpm'])} | {fmt(r['second']['macro_mae_bpm'])} | {fmt(r['second_minus_first_bpm'])} |")
display(Markdown('\n'.join(rows)))

## Limitations and inference
- The earlier legacy 15.40 BPM result mixes timing, ROI, reference and processing errors; it cannot isolate MRF failure.
- Corrected processing is implemented and tested, but this experiment does **not** establish successful breathing segmentation or reproduce published performance.
- Four people with overlapping windows provide very limited independent evidence. Still/moving labels assume a 60-second boundary.
- The paper uses 15 participants, specific camera views and a BIOPAC waveform. Our 8-bit recordings and event labels cannot yield its waveform correlation or equivalent cycle accuracy.
- Gaussian/Potts and tracking choices remain reproduction assumptions. High temporal SD can also reflect motion, camera changes or noise; MRF spatial smoothness does not identify respiration by itself.
- Missing source frames cannot be reconstructed by changing timestamps or duplicating frames. Relaxing validity thresholds after observing this result would define a new exploratory experiment.

## What to do next, in order
1. Audit freeze timing against original acquisition files and reference timestamps. Recover uninterrupted source recordings if available; preserve the current run as the first corrected experiment.
2. Predeclare a clean-segment experiment: obtain continuous still recordings with at least four seconds of warm-up followed by 20 seconds of valid synchronized data. Use the same KLT-only versus KLT+MRF comparison and the same rate estimators. Report all exclusions.
3. Annotate nasal landmarks and breathing-associated pixels independently of rate errors. Quantify alignment error and segmentation overlap, separating tracking, segmentation and downstream rate estimation failures.
4. Confirm underspecified MRF settings from author code or correspondence; assess alternatives only in a separate development set. Do not tune on held-out rate labels.
5. Acquire paper-compatible views, radiometric recordings, repeated subjects/sessions and a synchronized respiratory waveform for a full replication.

**Presentation conclusion:** We corrected the pipeline toward Kwon's method, including dynamic BAF/BAP/MRF and inverse stabilization. The next controlled experiment revealed insufficient valid temporal coverage, so MRF accuracy remains unmeasured. The immediate research priority is valid continuous data and independent segmentation validation, not additional neural-model tuning.

## 16. Short supported segments: exploratory next experiment
**Executed 31 October 2026.** The 20-second Kwon-style MRF evaluation remains at zero eligible windows. This chapter asks whether the already extracted segmented signal helps rate estimation on 12-second portions with adequate support. It does **not** replace the original 20-second result.

The window, stride, estimators, inclusion rules, and interpretation were recorded before calculating errors in `docs/short_segment_protocol.md`. We inspected valid-run durations, not rate errors, to choose 12 seconds.

In [ ]:
from pathlib import Path
import json, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
root = Path.cwd()
if not (root / 'thermal_face').exists(): root = root.parent
import sys
if str(root) not in sys.path: sys.path.insert(0, str(root))
folder = root / 'artifacts/kwon_experiment'
display(Markdown((root / 'docs/short_segment_protocol.md').read_text(encoding='utf-8')))

## 1. Diagnose the available source support
The source recordings repeatedly contain pixel-identical frame runs. We keep their timestamps and mark those frames missing. The code below counts those runs and the longest uninterrupted period where the segmented trace is valid. The recordings alone do not establish why the freezes occur.

In [ ]:
people = ['Joao', 'Claudio', 'Manuel', 'Jaime']
rows = ['| Person | Frozen runs | Frozen frames | Longest valid MRF span (s) | KLT failure frames |', '|---|---:|---:|---:|---:|']
for person in people:
    with np.load(folder / 'diagnostics' / f'{person}.npz') as d:
        t, frozen, track = d['seconds'], d['frozen'], d['track_ok']
    with np.load(folder / 'klt_mrf' / f'{person}.npz') as z:
        valid = z['valid']
    starts = np.where(valid & ~np.r_[False, valid[:-1]])[0]
    ends = np.where(valid & ~np.r_[valid[1:], False])[0]
    longest = max((t[e]-t[s] for s, e in zip(starts, ends)), default=0)
    frozen_runs = int(np.count_nonzero(frozen & ~np.r_[False, frozen[:-1]]))
    rows.append(f'| {person} | {frozen_runs} | {int(frozen.sum())} | {longest:.2f} | {int((~track).sum())} |')
display(Markdown('\n'.join(rows)))

## 2. Run the locked 12-second paired evaluation
We use the existing corrected KLT-only and KLT+MRF traces. Windows start every two seconds. Both methods must have an eligible signal and a valid marker-1 reference for the same person and start time. The code saves all candidates, predictions, exclusions, and hashes.

In [ ]:
from thermal_face.evaluate_short_segments import run
report = run(folder, root / 'artifacts/short_segment_experiment/evaluation.json')
def show(value): return 'undefined' if value is None else f'{value:.3f}'
rows = ['| Condition | Estimator | Paired windows | KLT only MAE | KLT+MRF MAE | MRF minus KLT |', '|---|---|---:|---:|---:|---:|']
for condition in ['still', 'overall', 'moving', 'transition']:
    for estimator in ['spectral', 'peak_count']:
        r = report['comparisons'][estimator][condition]
        rows.append(f"| {condition} | {estimator} | {r['common_windows']} | {show(r['first']['macro_mae_bpm'])} | {show(r['second']['macro_mae_bpm'])} | {show(r['second_minus_first_bpm'])} |")
display(Markdown('\n'.join(rows)))
for variant, details in report['variants'].items():
    print(variant, 'eligible', details['eligible_windows'], '/', details['candidate_windows'],
          'signal exclusions', details['signal_exclusions'],
          'reference exclusions', details['reference_exclusions'])

## 3. Inspect each participant, not just the overall average
Still-window spectral error falls for all four participants, but the number of overlapping windows varies. The plot gives each person one pair of bars. Four participants, rather than 14 windows, is the independent sample size.

In [ ]:
r = report['comparisons']['spectral']['still']
names = people
a = [r['first']['per_subject_mae_bpm'][p] for p in names]
b = [r['second']['per_subject_mae_bpm'][p] for p in names]
counts = {p: sum(k[0] == p for k in r['window_keys']) for p in names}
x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.bar(x-.18, a, .36, label='KLT ROI only', color='#758b9a')
ax.bar(x+.18, b, .36, label='KLT + MRF', color='#148084')
ax.set_xticks(x, [f'{p} (n={counts[p]})' for p in names])
ax.set_ylabel('Still-window spectral MAE (breaths/min)')
ax.set_title('Exploratory 12-second paired comparison')
ax.grid(axis='y', alpha=.2); ax.legend(); fig.tight_layout()
plot_path = root / 'artifacts/short_segment_experiment/per_person_still.png'
fig.savefig(plot_path, dpi=160); plt.close(fig)
display(Image(filename=str(plot_path)))

## 4. Prepare masks for independent review
The images below are chosen from the middle of each person's **longest valid MRF run**, using validity only. The middle column shows the dynamic mask and the last overlays it on the aligned crop. These are review prompts, not ground-truth annotations. A high-variation region may also reflect movement or image noise; only independent labels can establish whether the selected pixels are truly breathing-associated.

In [ ]:
import cv2
from thermal_face.kwon import crop_warp
boxes = json.loads((root / 'config/nose_boxes.json').read_text())
fig, axes = plt.subplots(4, 3, figsize=(9.5, 11))
review = []
for row, person in enumerate(people):
    paths = sorted((root / 'Manuel_clean' / person).glob('frame_*.png'))
    with np.load(folder / 'diagnostics' / f'{person}.npz') as d, np.load(folder / 'klt_mrf' / f'{person}.npz') as z:
        valid, t = z['valid'], z['seconds']
        starts = np.where(valid & ~np.r_[False, valid[:-1]])[0]
        ends = np.where(valid & ~np.r_[valid[1:], False])[0]
        s, e = max(zip(starts, ends), key=lambda se: t[se[1]]-t[se[0]])
        idx = int((s+e)//2)
        frame = cv2.imread(str(paths[idx]), cv2.IMREAD_GRAYSCALE)
        crop, support = crop_warp(frame, d['matrices'][idx], boxes[person])
        mask = np.unpackbits(d['klt_mrf_packed_masks'][idx], count=crop.size).reshape(crop.shape).astype(bool)
        assert valid[idx] and mask.any() and mask.shape == crop.shape
        base = np.repeat((crop/255.0)[..., None], 3, axis=2)
        overlay = base.copy(); overlay[mask] = .55*base[mask] + .45*np.array([0.0, 1.0, .85])
        for col, (image, title, cmap) in enumerate([(crop, 'Aligned crop', 'gray'), (mask, 'MRF mask', 'gray'), (overlay, 'Overlay', None)]):
            axes[row, col].imshow(image, cmap=cmap, vmin=0 if col < 2 else None, vmax=255 if col == 0 else (1 if col == 1 else None))
            axes[row, col].set_title(f'{person}, {t[idx]:.1f}s: {title}', fontsize=10)
            axes[row, col].axis('off')
        review.append({'person': person, 'frame_index_zero_based': idx, 'seconds': float(t[idx]),
                       'mask_fraction': float(mask.mean())})
fig.tight_layout()
review_path = root / 'artifacts/short_segment_experiment/mask_review.png'
fig.savefig(review_path, dpi=150); plt.close(fig)
(root / 'artifacts/short_segment_experiment/mask_review_selection.json').write_text(json.dumps(review, indent=2))
display(Image(filename=str(review_path)))
print('Selection:', review)

## 5. Interpretation, limitations, and next step
On the prespecified follow-up endpoint, still spectral participant-macro MAE changes from **4.874 to 2.564 breaths/min** (14 paired windows, four people). Still peak-count MAE changes from **2.856 to 2.111**. The moving condition does not improve: spectral changes from 12.543 to 12.591, and peak count from 6.752 to 7.585. This is a small exploratory result on 12-second, overlapping windows. It is not evidence that the mask is anatomically correct or that the original 20-second experiment succeeded. The support-based review images show substantial non-nasal selections: face edges for Joao and Manuel, an eye and edge for Claudio, and nasal/cheek areas for Jaime. Joao's aligned crop itself misses the nose. These images make independent localization and mask annotation a necessary next gate even though still-window rate error improved.

**Limitations:** the four recordings differ from Kwon's data and lack a synchronized waveform and pixel-level mask labels. Original camera/acquisition logs are unavailable, so the cause of repeated frozen frames is unknown. The movement boundary and marker interpretation still need verification. MRF settings partly reflect documented implementation assumptions. Short windows provide a coarse breathing-rate estimate. Many candidates are excluded: 29/222 MRF windows pass both signal and reference checks, compared with 126/222 KLT-only windows. A score without its coverage would be misleading.

**Next:** verify and recover continuous source recordings if possible; have a reviewer annotate nasal landmarks and breathing-associated regions on the saved mask examples without looking at rate errors; then rerun a prespecified paired **20-second** comparison on longer synchronized recordings. Keep the original 0/84 result and this short-window follow-up separate. No thresholds were tuned against these errors.